## SETUP AE

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
from scipy.fft import fft, fftfreq
from scipy import signal
from scipy.signal import savgol_filter
from scipy.stats import spearmanr, linregress

In [ ]:
# Util functions

def plot_emg(time_data,emg_data,col_name):
    plt.figure(figsize=(10, 6))

    # Plot for raw data
    #plt.subplot(1, 2, 1)  # 1 row, 2 columns, first subplot
    plt.plot(time_data, emg_data, label="EMG Data", color='blue')
    plt.xlabel("Time (s)")
    plt.ylabel("EMG (V)")
    plt.title(str(col_name))
    plt.grid(True)

    # Display the plot

    plt.tight_layout()
    plt.show()


def plot_emg_wlabels(x, y, name=None, label='Frequencies', gt_time=None):
    plt.figure(figsize=(12, 6))
    plt.plot(x, y)
    plt.title(str(label), fontsize=16) 
    plt.xlabel('Time (s)', fontsize=14)  
    plt.ylabel('Frequency (Hz)', fontsize=14)  
    plt.grid(True)
    plt.axvline(x=gt_time[0], color='black', linestyle='--', label='Non-Fatigue')
    plt.axvline(x=gt_time[1], color='black', linestyle='--', label='Transition-to-Fatigue')
    
    if name is not None:
        plt.savefig(name)
        print("saved at", name)
    plt.close()
    

def mean_frequency(power_spectrum, frequencies):
    """
    Calculate the mean frequency from the power spectrum.
    Args:
    - power_spectrum: Power spectrum from FFT.
    - frequencies: Corresponding frequency bins.
    
    Returns:
    - Mean frequency.
    """

    if len(power_spectrum) == 0 or np.sum(power_spectrum) == 0:
        print("-------- power spectrum is empty --------")
        return None  
    

    weighted_sum = np.sum(frequencies * power_spectrum)
    total_power = np.sum(power_spectrum)
    
    if total_power == 0:
        print( "------------ Total Power is zero --------- ")
        return None
    
    mean_freq = weighted_sum / total_power
    
    return mean_freq


def median_frequency(power_spectrum, frequencies):
    """
    Calculate the median frequency from the power spectrum.
    Args:
    - power_spectrum: Power spectrum from FFT.
    - frequencies: Corresponding frequency bins.
    
    Returns:
    - Median frequency.
    """
    cumulative_power = np.cumsum(power_spectrum)
    total_power = cumulative_power[-1] #last element retrives the total power
    
    # Find the frequency where the cumulative power reaches 50% of the total power
    median_idx = np.where(cumulative_power >= (total_power / 2))[0][0]

    return frequencies[median_idx]


def calculate_median_frequencies(signal, time, sampling_rate, window_size_seconds, step_size_seconds):
    """
    Calculate the median frequency using a sliding window approach.
    Args:
    - signal: The raw signal values.
    - sampling_rate: The sampling rate of the signal.
    - window_size_seconds: Duration of each window in seconds.
    - step_size_seconds: Step size for sliding the window in seconds.
    
    Returns:
    - List of median frequencies for each window.
    """

    window_size = int(window_size_seconds * sampling_rate)  # Number of samples in one window
    step_size = int(step_size_seconds * sampling_rate)      # Number of samples to slide the window
    num_windows = int((len(signal) - window_size) // step_size + 1)
    
    median_frequencies = []
    mean_frequencies = []
    window_time_center = []

    for i in range(num_windows):

        # Get the current window
        start_idx = i * step_size
        end_idx = start_idx + window_size
        window_signal = signal[start_idx:end_idx]
        window_time = time[start_idx:end_idx]

        # Calculate the center time for the current window
        center_time = window_time[int(len(window_time) // 2)]
        window_time_center.append(center_time)

        signal_mean = np.mean(signal)
        #plot(window_time,window_signal, name=filename, label='raw sEMG windows')

        # Perform FFT on the windowed signal
        fft_result = fft(window_signal-signal_mean)
        frequencies = fftfreq(len(window_signal), d=1/sampling_rate)
        power_spectrum = np.abs(fft_result)**2  # Power spectrum

        # Only consider positive frequencies (first half of FFT result)
        positive_frequencies = frequencies[:len(frequencies)//2]
        positive_power_spectrum = power_spectrum[:len(power_spectrum)//2]  

        # Calculate median frequency for this window
        med_freq = median_frequency(positive_power_spectrum, positive_frequencies)
        mean_freq = mean_frequency(positive_power_spectrum, positive_frequencies)
        median_frequencies.append(med_freq)
        mean_frequencies.append(mean_freq)


    return window_time_center, median_frequencies, mean_frequencies


# Function to normalize time by maximum duration
def normalize_time_to_max(raw_time, max_time):
    if raw_time[-1] < max_time:
        # Pad with NaNs to match the max_time
        padding = np.nan * np.ones(int(max_time - len(raw_time)))
        return np.concatenate((raw_time, padding))
    else:
        # Truncate if longer
        return raw_time
    

def plot_overlapping_data(time_data, muscle_data, metric, muscle_name):
    plt.figure(figsize=(20, 14))

    cmap = plt.get_cmap('tab20')  
    colors = [cmap(i) for i in range(13)]  

    line_styles = ['-', '--']  
    line_styles = line_styles * 7  # Repeat to match the number of subjects
    
    # Loop through subjects
    for subject_number in range(1, 14):  # Loop through all subjects (1 to 13)
        subject_key = f'subject_{subject_number}'
        
        for trial_index in range(len(time_data[subject_key])):  # Loop through trials for each subject
                    time = time_data[subject_key][trial_index]
                    data = muscle_data[subject_key][trial_index]
                    if len(time) == len(data):  # Ensure that time and data have matching lengths

                        smoothed_data = savgol_filter(data, window_length=8, polyorder=3)  # Adjust window_length and polyorder for best results

                        # Plot smoothed trend
                        plt.plot(time, smoothed_data, label=f'Subject {subject_number} - {muscle_name}', 
                                 color=colors[subject_number - 1], linestyle=line_styles[subject_number % len(line_styles)])
                    else:
                        print(f"Time and data lengths mismatch for subject {subject_number}, {metric}")

    # Set plot titles and labels once after all subjects are plotted
    #plt.title(f'{muscle_name} ', fontsize=18)
    plt.xlabel('Time (s)', fontsize=18)
    plt.ylabel(f'{metric} (Hz)', fontsize=18)
    plt.legend(loc='best', fontsize=14)  # Move legend outside the plot for clarity
    plt.grid(True)

    # Save the plot
    name = f"{metric}_{muscle_name.replace(' ', '_')}.png"
    filename = os.path.join(dir, name)
    plt.savefig(filename)
    plt.show()
    plt.close()
    print(f"Saved plot at {filename}")

    #associate muscle with trial
def get_prime_mover(trial):
    mapping = {1:1,2:0, # anterior
               3:3,4:2, # posterior
               5:0,6:1, # biceps
               7:2,8:3, # medius
               9:1,10:0, # anterior
               11:3,12:2} # posterior
    return mapping.get(trial,None)

def extract_trial_summary(time_arr, mnf_arr, mdf_arr, subject_number, trial, muscle_name, baseline_n=3):
    time_arr = np.array(time_arr)
    mnf_arr = np.array(mnf_arr)
    mdf_arr = np.array(mdf_arr)
    
    row = {
        'subject': subject_number,
        'trial': trial,
        'muscle': muscle_name,
        'duration_s': time_arr[-1] - time_arr[0],
        'n_windows': len(time_arr),
    }
    
    for name, arr in [('MNF', mnf_arr), ('MDF', mdf_arr)]:
        baseline = np.mean(arr[:baseline_n])
        final = np.mean(arr[-baseline_n:])
        rho, pval = spearmanr(time_arr, arr)
        slope, intercept, r, p, se = linregress(time_arr, arr)
        
        row[f'{name}_baseline'] = baseline
        row[f'{name}_final'] = final
        row[f'{name}_pct_change'] = (final - baseline) / baseline * 100
        row[f'{name}_spearman_rho'] = rho
        row[f'{name}_spearman_p'] = pval
        row[f'{name}_slope'] = slope
        row[f'{name}_r2'] = r**2
        row[f'{name}_cv'] = np.std(arr) / np.mean(arr)
    
    return row

def total_power(power_spectrum):
    return np.sum(power_spectrum)

def rms_windowed(signal, sampling_rate, window_size_seconds, step_size_seconds):
    window_size = int(window_size_seconds * sampling_rate)
    step_size = int(step_size_seconds * sampling_rate)
    num_windows = int((len(signal) - window_size) // step_size + 1)
    
    rms_values = []
    for i in range(num_windows):
        start = i * step_size
        end = start + window_size
        window = signal[start:end]
        rms_values.append(np.sqrt(np.mean(window**2)))
    return rms_values

def extract_gt_features(gt_times, total_duration):
    non_fatigue_end, transition_end = gt_times
    return {
        'gt_non_fatigue_duration': non_fatigue_end,
        'gt_transition_duration': transition_end - non_fatigue_end,
        'gt_non_fatigue_pct': non_fatigue_end / total_duration * 100,
        'gt_transition_pct': (transition_end - non_fatigue_end) / total_duration * 100,
    }

# Trial-to-muscle correspondence dictionary
trial_to_muscle = {
    1: "R DELTOID ANTERIOR",
    2: "L DELTOID ANTERIOR",
    3: "R DELTOID POSTERIOR",
    4: "L DELTOID POSTERIOR",
    5: "R BICEPS BRACHII",
    6: "L BICEPS BRACHII",
    7: "R DELTOID MEDIUS",
    8: "L DELTOID MEDIUS",
    9: "R DELTOID ANTERIOR C",
    10: "L DELTOID ANTERIOR C",
    11: "R DELTOID POSTERIOR C",
    12: "L DELTOID POSTERIOR C",
}

                

In [ ]:
dir = r'D:\smilecow\HUST\Nam_3\nckh\dataset\sEMG_data'
GT_dir = r'D:\smilecow\HUST\Nam_3\nckh\dataset\self_perceived_fatigue_index'

subject_numbers = range(1, 14)  # Adjust the range as needed

not_filtered = True
X_subjects = []
true_labels_subjects = []
time= []
EMG_index = [1,3,5,7]
time_index = [0,2,4,6]
sample_freq = 1259 #Hz

## nghich data ti

In [ ]:

# # Initialize a dictionary to store the data for each muscle
# # Initialize a dictionary to store the MNF/MDF data for each muscle, subject, and trial
# muscle_data_MNF = {f'muscle_{i}': {f'subject_{j}': [] for j in range(1, 14)} for i in range(1, 14)}
# muscle_data_MDF = {f'muscle_{i}': {f'subject_{j}': [] for j in range(1, 14)} for i in range(1, 14)}
# time_dic = {f'muscle_{i}': {f'subject_{j}': [] for j in range(1, 14)} for i in range(1, 14)}

# # Step 1: Find the maximum time across all subjects and trials
# max_time = 0

# # Loop through each subject
# for subject_number in subject_numbers:
#     emg_directory = os.path.join(dir, f'subject_{subject_number}')
#     #GT_directory = os.path.join(dir, f'Subject_{subject_number}')
    
#     emg_files = sorted([f for f in os.listdir(emg_directory) if f.endswith(".csv")], key=lambda x: int(re.findall(r'\d+', x)[-1]))
#     #GT_files = sorted([f for f in os.listdir(GT_directory) if f.endswith(".csv")], key=lambda x: int(re.findall(r'\d+', x)[-1]))

#     data_subject = []  # List to store data for current subject
#     labels_subject = []  # List to store true labels for current subject

#     for emg_file in zip(emg_files):
#         emg_path = os.path.join(emg_directory, emg_file[0])

#         file = pd.read_csv(emg_path, delimiter=',', header=0)
#         file_values = file.values
#         columns_names = file.columns
#         trial = int(re.findall(r'\d+', emg_file[0])[-1])
    
#         # Loop through EMG and time indices
#         index = get_prime_mover(trial)
#         print("trial", trial, index)
        

#         emg_data = file_values[:, EMG_index[index]]
#         time_data = file_values[:, time_index[index]]
#         col_name = columns_names[EMG_index[index]]
#         #plot_emg(time_data, emg_data,col_name)

#         if not_filtered:
#             fc1, fc2 = 20, 450
#             b, a = signal.butter(4, [fc1, fc2], btype='bandpass', fs=sample_freq)
#             filtered_emg = signal.filtfilt(b, a, emg_data)
#             emg_data = filtered_emg


#         # Define window and step size
#         window_size_seconds = 4  # 4-second window
#         step_size_seconds = 2    # 50% overlap
#         #Overlapping windows help reduce abrupt changes between segments and provide a more continuous analysis of the signal.

#         # Calculate median frequencies
#         window_time_centers, median_frequencies, mean_frequencies = calculate_median_frequencies(emg_data,time_data, sample_freq, window_size_seconds, step_size_seconds )

#         ## ---- Single emg  plots --- ##
#         #plot_emg(window_time_centers,median_frequencies,("MDF" + col_name))
#         #plot_emg(window_time_centers,mean_frequencies,("MNF" + col_name))

#         max_time = max(max_time, window_time_centers[-1])  # Update the maximum time
        

#         # Store the data by muscle

#         muscle_data_MNF[f'muscle_{trial}'][f'subject_{subject_number}'].append(mean_frequencies)
#         muscle_data_MDF[f'muscle_{trial}'][f'subject_{subject_number}'].append(median_frequencies)
#         time_dic[f'muscle_{trial}'][f'subject_{subject_number}'].append(window_time_centers)

# for trial in range(1, 13):
#     # Get the corresponding muscle name from the trial index
#     muscle_name = trial_to_muscle.get(trial)
#     print(f"Plotting data for {muscle_name}")

#     # Check if there is any data to plot
#     if time_dic[f'muscle_{trial}'] and muscle_data_MNF[f'muscle_{trial}']:
#         # Plot MNF
#         plot_overlapping_data(time_dic[f'muscle_{trial}'], muscle_data_MNF[f'muscle_{trial}'], 
#                               'Mean Frequency', f'{muscle_name}')

#     if time_dic[f'muscle_{trial}'] and muscle_data_MDF[f'muscle_{trial}']:
#         # Plot MDF
#         plot_overlapping_data(time_dic[f'muscle_{trial}'], muscle_data_MDF[f'muscle_{trial}'], 
#                               'Median Frequency', f'{muscle_name}')
#     else:
#         print(f"No data to plot for muscle {muscle_name}")






In [ ]:
# ## Frequency analysis  - MDF and MNF
# # PLOT a graph with the MDF values of 1 cycle
# # velocity was 30 bpm, meaning that there was a beat every 2 seconds. 
# #therefore, the cycle was 2 seconds and the window size should also be 

# from scipy import signal

# base_directory = r'D:\smilecow\HUST\Nam_3\nckh\dataset'

# EMG_index = [1,3,5,7]
# time_index = [0,2,4,6]
# sample_freq = 1259 #Hz

# for subject_number in range(1, 14): 

#   path_directory = os.path.join(base_directory, 'sEMG_data', f'subject_{subject_number}')
#   label_directory = os.path.join(base_directory, 'self_perceived_fatigue_index', f'subject_{subject_number}')
#   global out_directory
#   out_directory = os.path.join(base_directory, '','frequency_analysis', 'per_trial',f'subject_{subject_number}')

#   isExist = os.path.exists(out_directory)
#   if not isExist:


#     os.makedirs(out_directory)


#   files = [file_name for file_name in os.listdir(path_directory) if file_name.endswith("csv")]
#   files = sorted(files, key=lambda x: int(re.findall(r'\d+', x)[-1]))

#   label_files = [file_name for file_name in os.listdir(label_directory) if file_name.endswith("csv")]
#   label_files = sorted(label_files, key=lambda x: int(re.findall(r'\d+', x)[-1]))
  
#   for file_raw, file_down in zip(files, label_files):
#         # Read the CSV file
        
#         raw_files_path = os.path.join(path_directory, file_raw)
#         label_files_path = os.path.join(label_directory, file_down)

#         trial = re.findall(r'\d+', file_raw)
#         print("file", file_raw, "trial", int(trial[0]))
#         index = get_prime_mover(int(trial[0]))


#         raw = pd.read_csv(raw_files_path, delimiter=',', header=0)
#         file_values = raw.values
#         raw_time =  file_values[:, time_index[index]]
#         raw_val =  file_values[:, EMG_index[index]]
#         if not_filtered:
#           fc1, fc2 = 20, 450
#           b, a = signal.butter(4, [fc1, fc2], btype='bandpass', fs=sample_freq)
#           filtered_emg = signal.filtfilt(b, a, raw_val)
#           raw_val = filtered_emg

#         label = pd.read_csv(label_files_path, delimiter=',', header=0)
#         label_time = label.iloc[:, 0]  
#         label_values = label.iloc[:, 1]

#         raw_values = np.array(raw_val)
#         if np.isnan(raw_val).any():
#           print("NaN values found in the signal. Replacing NaN with zeros.")
    
#         if np.isinf(raw_val).any():
#             print("Inf values found in the signal. Replacing Inf with zeros.")

#         raw_values = np.where(np.isnan(raw_val) | np.isinf(raw_val), 0, raw_val)

#       # ----- Segment 0: From the beginning to the last 0 in label_values
#         idx_last_0 = label_values[label_values == 0].index[-1]
#         segment_0_label_time = label_time[idx_last_0]
#         # Find the index in raw_time that corresponds to the nearest time to segment_1_label_time
#         nearest_idx = (np.abs(raw_time - segment_0_label_time)).argmin()
#         # Nearest time and values in raw_time
#         nearest_time = raw_time[nearest_idx]

#         # ------- Segment 1: From the the fisrt 1 to the last 1 in label_values
#         idx_last_1 = label_values[label_values == 1].index[-1]
#         segment_1_label_time = label_time[idx_last_1]
#         nearest_idx_1 = (np.abs(raw_time - segment_1_label_time)).argmin()
#         nearest_time_1 = raw_time[nearest_idx_1]

#         gt_times = [nearest_time, nearest_time_1]
  
#         # Define window and step size FOR MDF AND MNF ANALYSIS
#         window_size_seconds = 4  # 4-second window
#         step_size_seconds = 2    # 4-second step (50% overlap) 
#         #Overlapping windows help reduce abrupt changes between segments and provide a more continuous analysis of the signal.

#         # Calculate median frequencies
#         window_time_centers, median_frequencies,mean_frequencies = calculate_median_frequencies(raw_values,raw_time, sample_freq, window_size_seconds, step_size_seconds )

#         print(f"Median Frequencies for {file_raw}: {median_frequencies}")
#         freq_MDF_name = os.path.join(out_directory,f'MDF_{(file_raw.replace(".csv",".png"))}')
#         #id = np.arange(0,len(median_frequencies),1)
#         plot_emg_wlabels(window_time_centers,median_frequencies,name=freq_MDF_name, label="median frequencies", gt_time=gt_times)


#         #n = np.arange(0,len(mean_frequencies),1)
#         freq_name = os.path.join(out_directory,f'MNF_{(file_raw.replace(".csv",".png"))}')
#         plot_emg_wlabels(window_time_centers,mean_frequencies, name = freq_name, label='mean frequencies', gt_time=gt_times)
#         print ("---")

In [ ]:
# # ================================================================
# # PHASE B FIX — Thêm TP & RMS vào bảng chỉ số chất lượng
# # ----------------------------------------------------------------
# # Phụ thuộc: Cell 0-2 (imports, utility functions, dir config).
# # Output   : dataset/quality_metrics_v2.csv  (37 cột)
# # ================================================================
# from scipy.stats import spearmanr, linregress

# def extract_summary_stats(time_arr, feat_arr, prefix, baseline_n=3):
#     """Tính baseline/final/pct_change/slope/rho/r2/cv cho 1 chuỗi đặc trưng."""
#     feat_arr  = np.array(feat_arr, dtype=float)
#     time_arr  = np.array(time_arr, dtype=float)
#     base_n    = min(baseline_n, len(feat_arr))
#     baseline  = np.nanmean(feat_arr[:base_n])
#     final_val = np.nanmean(feat_arr[-base_n:])
#     pct       = (final_val - baseline) / (abs(baseline) + 1e-9) * 100
#     rho, _    = spearmanr(time_arr, feat_arr)
#     slope, _, r, _, _ = linregress(time_arr, feat_arr)
#     cv        = np.nanstd(feat_arr) / (abs(np.nanmean(feat_arr)) + 1e-9) * 100
#     return {
#         f'{prefix}_baseline'    : baseline,
#         f'{prefix}_final'       : final_val,
#         f'{prefix}_pct_change'  : pct,
#         f'{prefix}_spearman_rho': rho,
#         f'{prefix}_slope'       : slope,
#         f'{prefix}_r2'          : r**2,
#         f'{prefix}_cv'          : cv,
#     }

# def calculate_all_features_v2(sig, time_arr, sampling_rate,
#                                window_size_s=4, step_size_s=2):
#     """
#     Trả về (t_centers, mnf_list, mdf_list, tp_list, rms_list)
#     cho mỗi sliding window — tính 1 lần FFT duy nhất cho tất cả đặc trưng.
#     """
#     win = int(window_size_s * sampling_rate)
#     stp = int(step_size_s  * sampling_rate)
#     n_win = (len(sig) - win) // stp + 1
#     sig_mean = np.mean(sig)

#     t_centers, mnf_l, mdf_l, tp_l, rms_l = [], [], [], [], []

#     for i in range(n_win):
#         s, e  = i * stp, i * stp + win
#         w_sig  = sig[s:e]
#         w_time = time_arr[s:e]
#         t_centers.append(w_time[len(w_time) // 2])
#         rms_l.append(np.sqrt(np.mean(w_sig**2)))

#         fft_res = fft(w_sig - sig_mean)
#         freqs   = fftfreq(len(w_sig), d=1.0/sampling_rate)
#         p_spec  = np.abs(fft_res)**2
#         pos     = freqs[:len(freqs)//2]
#         pos_p   = p_spec[:len(p_spec)//2]
#         band    = (pos >= 20) & (pos <= 450)

#         tp_l.append(total_power(pos_p[band]))
#         mdf_l.append(median_frequency(pos_p[band], pos[band]))
#         mnf_l.append(mean_frequency(pos_p[band], pos[band]))

#     return t_centers, mnf_l, mdf_l, tp_l, rms_l

# # ── Vòng lặp chính ──────────────────────────────────────────────
# rows_v2 = []
# dataset_dir = os.path.dirname(dir)   # d:/…/dataset

# for subject_number in subject_numbers:
#     path_dir = os.path.join(dir, f'subject_{subject_number}')
#     if not os.path.exists(path_dir):
#         continue
#     files = sorted([f for f in os.listdir(path_dir) if f.endswith(".csv")],
#                    key=lambda x: int(re.findall(r'\d+', x)[-1]))

#     for file_raw in files:
#         trial = int(re.findall(r'\d+', file_raw)[0])
#         muscle_name = trial_to_muscle.get(trial)
#         if muscle_name is None:
#             continue

#         idx = get_prime_mover(trial)
#         raw = pd.read_csv(os.path.join(path_dir, file_raw))
#         vals = raw.values
#         raw_time = vals[:, time_index[idx]]
#         raw_emg  = vals[:, EMG_index[idx]]

#         b_f, a_f = signal.butter(4, [20, 450], btype='bandpass', fs=sample_freq)
#         filtered = signal.filtfilt(b_f, a_f, np.nan_to_num(raw_emg, nan=0.0))

#         t_c, mnf_v, mdf_v, tp_v, rms_v = calculate_all_features_v2(
#             filtered, raw_time, sample_freq, 4, 2)

#         if len(t_c) < 3:
#             continue

#         t_arr = np.array(t_c)
#         row = {'subject': subject_number, 'trial': trial, 'muscle': muscle_name,
#                'movement_type': 'complex' if ' C' in muscle_name else 'uni-articular',
#                'n_windows': len(t_c), 'duration_s': float(t_arr[-1] - t_arr[0])}
#         row.update(extract_summary_stats(t_arr, mnf_v, 'MNF'))
#         row.update(extract_summary_stats(t_arr, mdf_v, 'MDF'))
#         row.update(extract_summary_stats(t_arr, tp_v,  'TP'))
#         row.update(extract_summary_stats(t_arr, rms_v, 'RMS'))
#         rows_v2.append(row)
#     print(f"  Subject {subject_number}: OK ({len([r for r in rows_v2 if r['subject']==subject_number])} trials)")

# v2_df = pd.DataFrame(rows_v2)

# # ── Gắn cờ Outlier (logic ĐẢO NGƯỢC giữa MNF/MDF và TP/RMS) ────
# v2_df['is_outlier_MNF'] = v2_df['MNF_spearman_rho'] >= 0   # MNF phải giảm → rho < 0
# v2_df['is_outlier_TP']  = v2_df['TP_spearman_rho']  <= 0   # TP  phải tăng → rho > 0
# v2_df['is_outlier_RMS'] = v2_df['RMS_spearman_rho'] <= 0   # RMS phải tăng → rho > 0
# v2_df['n_signals_outlier'] = (v2_df['is_outlier_MNF'].astype(int) +
#                                 v2_df['is_outlier_TP'].astype(int) +
#                                 v2_df['is_outlier_RMS'].astype(int))

# out_v2 = os.path.join(dataset_dir, 'quality_metrics_v2.csv')
# v2_df.to_csv(out_v2, index=False)
# print(f"\n[Phase B Fix] Saved {len(v2_df)} rows × {len(v2_df.columns)} cols → {out_v2}")

# all_agree = (v2_df['n_signals_outlier'] == 0).sum()
# print(f"  Tất cả 3 tín hiệu đúng chiều (MNF↓, TP↑, RMS↑): "
#       f"{all_agree}/{len(v2_df)} trials ({all_agree/len(v2_df)*100:.1f}%)")
# print("\nTop 5 cột TP/RMS mới:")
# print(v2_df[['subject','trial','muscle','TP_spearman_rho','RMS_spearman_rho',
#              'TP_pct_change','RMS_pct_change']].head())


In [ ]:
# # ============================================================
# # PHASE C: Khai thác Ground Truth (self_perceived_fatigue_index)
# # Câu hỏi kiểm chứng: "sEMG có phát hiện mỏi SỚM HƠN
# # cảm nhận chủ quan của subject không?"
# # ============================================================

# gt_rows = []

# for subject_number in subject_numbers:
#     path_dir  = os.path.join(os.path.dirname(dir), 'sEMG_data',
#                              f'subject_{subject_number}')
#     label_dir = os.path.join(os.path.dirname(dir), 'self_perceived_fatigue_index',
#                              f'subject_{subject_number}')
#     if not os.path.exists(path_dir) or not os.path.exists(label_dir):
#         print(f"  Skip subject_{subject_number}: thư mục không tồn tại")
#         continue

#     emg_files = sorted([f for f in os.listdir(path_dir) if f.endswith(".csv")],
#                        key=lambda x: int(re.findall(r'\d+', x)[-1]))
#     lbl_files = sorted([f for f in os.listdir(label_dir) if f.endswith(".csv")],
#                        key=lambda x: int(re.findall(r'\d+', x)[-1]))

#     for file_emg, file_lbl in zip(emg_files, lbl_files):
#         trial = int(re.findall(r'\d+', file_emg)[0])
#         muscle_name = trial_to_muscle.get(trial)
#         if muscle_name is None:
#             continue

#         index = get_prime_mover(trial)

#         raw   = pd.read_csv(os.path.join(path_dir, file_emg))
#         vals  = raw.values
#         raw_time = vals[:, time_index[index]]
#         raw_emg  = vals[:, EMG_index[index]]

#         # Lọc bandpass
#         fc1, fc2 = 20, 450
#         b_f, a_f = signal.butter(4, [fc1, fc2], btype='bandpass', fs=sample_freq)
#         clean_emg = np.nan_to_num(raw_emg, nan=0.0, posinf=0.0, neginf=0.0)
#         filtered  = signal.filtfilt(b_f, a_f, clean_emg)

#         # Đọc ground truth labels
#         lbl = pd.read_csv(os.path.join(label_dir, file_lbl))
#         lbl_time = lbl.iloc[:, 0]
#         lbl_vals = lbl.iloc[:, 1]

#         try:
#             non_fatigue_end = lbl_time[lbl_vals[lbl_vals == 0].index[-1]]
#             transition_end  = lbl_time[lbl_vals[lbl_vals == 1].index[-1]]
#         except (IndexError, KeyError):
#             print(f"  Skip subj={subject_number} trial={trial}: GT label lỗi")
#             continue

#         total_dur = float(raw_time[-1] - raw_time[0])
#         gt_feats  = extract_gt_features([non_fatigue_end, transition_end], total_dur)

#         # Tính MNF theo sliding window
#         t_centers, _, mnf_vals = calculate_median_frequencies(
#             filtered, raw_time, sample_freq, 4, 2)
#         if len(t_centers) < 3:
#             continue

#         t_arr   = np.array(t_centers)
#         mnf_arr = np.array(mnf_vals)

#         base_n = min(3, len(mnf_arr))
#         mnf_baseline = np.mean(mnf_arr[:base_n])

#         # MNF tại thời điểm GT non_fatigue_end
#         idx_gt     = min(np.searchsorted(t_arr, non_fatigue_end), len(mnf_arr)-1)
#         mnf_at_gt  = float(mnf_arr[idx_gt])
#         mnf_drop_pct = (mnf_at_gt - mnf_baseline) / mnf_baseline * 100                        if mnf_baseline != 0 else float('nan')

#         gt_rows.append({
#             'subject'                  : subject_number,
#             'trial'                    : trial,
#             'muscle'                   : muscle_name,
#             'movement_type'            : 'complex' if ' C' in muscle_name else 'uni-articular',
#             'total_duration_s'         : total_dur,
#             'MNF_baseline_Hz'          : mnf_baseline,
#             'MNF_at_gt_nonfatigue_end' : mnf_at_gt,
#             'MNF_drop_pct_at_gt'       : mnf_drop_pct,
#             **gt_feats,
#         })

# gt_df = pd.DataFrame(gt_rows)
# gt_csv = os.path.join(dir, 'gt_crossref.csv')
# gt_df.to_csv(gt_csv, index=False)
# print(f"[Phase C] Saved {len(gt_df)} rows → {gt_csv}\n")

# print("─── MNF đã giảm bao nhiêu tại thời điểm chủ quan bắt đầu mỏi? ───")
# print("(Âm = sEMG thấy mỏi trước cảm nhận; dương = sEMG chậm hơn cảm nhận)")
# print(gt_df.groupby('muscle').agg(
#     MNF_drop_mean        =('MNF_drop_pct_at_gt','mean'),
#     MNF_drop_std         =('MNF_drop_pct_at_gt','std'),
#     gt_nonfatigue_pct_mean=('gt_non_fatigue_pct','mean'),
# ).round(2).to_string())

# print("\n─── Thống kê theo subject ───")
# print(gt_df.groupby('subject').agg(
#     tot_dur_mean        =('total_duration_s','mean'),
#     gt_nonfat_pct_mean  =('gt_non_fatigue_pct','mean'),
#     MNF_drop_mean       =('MNF_drop_pct_at_gt','mean'),
# ).round(2).to_string())


## pipeline

In [ ]:
# ================================================================
# PHASE E — Bước 1: Validation phân đoạn rep
# Subject 1, 5, 10 × Trial 5 (R Biceps Brachii)
# ================================================================
from scipy.signal import savgol_filter, find_peaks
import matplotlib.pyplot as plt

# ── Hàm phân đoạn (inverted-signal valley detection) ──────────────
def segment_reps(emg_filtered, time_arr, fs,
                 min_rep_s=1.2, max_rep_s=4.5, smooth_ms=200):
    """
    Phân đoạn rep bằng đường bao RMS + valley detection cải tiến.
    Tham số: min=1.2s / max=4.5s (30BPM baseline + velocity loss).
    Valley: tìm trực tiếp trên -smooth để tránh peaks giả sát nhau.
    Returns: segments [(start, end)], peaks_idx, valleys_idx
    """
    win = int(smooth_ms * 1e-3 * fs)
    # Vectorized RMS envelope
    sig_sq  = np.pad(emg_filtered**2, win//2, mode='edge')
    cumsum  = np.cumsum(sig_sq)
    rms_env = np.sqrt((cumsum[win:] - cumsum[:-win]) / win)

    sg_win = max(win * 3, 11)
    if sg_win % 2 == 0: sg_win += 1
    smooth = savgol_filter(rms_env, window_length=sg_win, polyorder=3)

    min_dist = int(min_rep_s * fs)
    prom     = np.std(smooth) * 0.5

    peaks,   _ = find_peaks( smooth, distance=min_dist, prominence=prom)
    valleys, _ = find_peaks(-smooth, distance=min_dist, prominence=prom)

    boundary = sorted(valleys)
    if not boundary:
        return [], peaks, valleys

    all_bounds = [0] + list(boundary) + [len(emg_filtered)-1]
    segments = [(s, e) for s, e in zip(all_bounds[:-1], all_bounds[1:])
                if min_rep_s <= (time_arr[e] - time_arr[s]) <= max_rep_s]

    return segments, peaks, valleys, smooth

# ── Validation: vẽ overlay cho 3 subjects ─────────────────────────
VALIDATE_SUBJECTS = [1, 5, 10]
VALIDATE_TRIAL    = 5   # R Biceps Brachii
muscle_trial5     = trial_to_muscle.get(VALIDATE_TRIAL, 'R Biceps Brachii')

fig, axes = plt.subplots(len(VALIDATE_SUBJECTS), 1,
                          figsize=(18, 5 * len(VALIDATE_SUBJECTS)), sharex=False)
fig.suptitle(f'Validation: Rep Segmentation — Trial {VALIDATE_TRIAL} ({muscle_trial5})',
             fontsize=14, fontweight='bold')

for ax, subj in zip(axes, VALIDATE_SUBJECTS):
    path_dir = os.path.join(dir, f'subject_{subj}')
    f_csv    = f'trial_{VALIDATE_TRIAL}.csv'
    if not os.path.exists(os.path.join(path_dir, f_csv)):
        ax.set_title(f'Subject {subj} — FILE NOT FOUND'); continue

    idx = get_prime_mover(VALIDATE_TRIAL)
    raw = pd.read_csv(os.path.join(path_dir, f_csv)).values
    raw_time = raw[:, time_index[idx]]
    raw_emg  = raw[:, EMG_index[idx]]

    b, a      = signal.butter(4, [20, 450], btype='bandpass', fs=sample_freq)
    filtered  = signal.filtfilt(b, a, np.nan_to_num(raw_emg, nan=0.0))

    result = segment_reps(filtered, raw_time, sample_freq)
    if len(result) == 4:
        segments, peaks, valleys, smooth = result
    else:
        ax.set_title(f'Subject {subj} — Segmentation failed'); continue

    # Plot
    ax.plot(raw_time, filtered, color='#aaaaaa', lw=0.5, alpha=0.7, label='sEMG filtered')
    # Normalize smooth for overlay
    env_scaled = smooth / (smooth.max() + 1e-9) * np.abs(filtered).max()
    ax.plot(raw_time[:len(smooth)], env_scaled, color='orange', lw=1.5, label='RMS envelope (smooth)')

    # Valley lines (rep boundaries)
    for v in valleys:
        if v < len(raw_time):
            ax.axvline(x=raw_time[v], color='green', lw=1.2, alpha=0.8)

    # Peak markers
    peak_times = [raw_time[p] for p in peaks if p < len(raw_time)]
    peak_vals  = [env_scaled[p] for p in peaks if p < len(smooth)]
    ax.scatter(peak_times, peak_vals, color='red', s=40, zorder=5, label='Peaks')

    n_reps = len(segments)
    dur    = raw_time[-1] - raw_time[0]
    ax.set_title(f'Subject {subj} — {n_reps} reps detected (duration={dur:.1f}s)', fontsize=11)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('sEMG (a.u.)')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.2)
    print(f"  Subject {subj}: {n_reps} reps | durations: "
          + str([f'{raw_time[e]-raw_time[s]:.1f}s' for s,e in segments]))

plt.tight_layout()
val_fig = os.path.join(os.path.dirname(dir), f'rep_segmentation_validation_trial{VALIDATE_TRIAL}.png')
plt.savefig(val_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f"\nSaved validation plot → {val_fig}")
print("\n>>> KIỂM TRA đồ thị trước khi tiếp tục! <<<")
print("    Nếu rep boundaries (vlines xanh) KHÔNG khớp với chu kỳ cơ → điều chỉnh prominence.")


In [ ]:
# ================================================================
# PHASE E — Bước 2 & 3: Full batch per-cycle + N1 normalization
# Phụ thuộc: Cell E-validation đã xác nhận segmentation ổn.
# Output: data/processed/per_cycle_features.csv
# ================================================================

def extract_per_cycle_features(emg_filtered, time_arr, fs,
                                rep_segments, subject, trial, muscle):
    """
    Với mỗi rep: chia 3 phần → tính MNF/MDF/TP/RMS → lấy max/min/mean.
    FCF = rep_idx / N_total; RIR = N_total - rep_idx (clip ≤10 theo [C1]).
    """
    N = len(rep_segments)
    if N == 0:
        return pd.DataFrame()

    rows = []
    for k, (s, e) in enumerate(rep_segments):
        seg  = emg_filtered[s:e]
        tseg = time_arr[s:e]
        if len(seg) < int(fs * 0.5):
            continue

        L = len(seg) // 3
        if L < 16:
            continue
        thirds = [seg[:L], seg[L:2*L], seg[2*L:]]

        feat = {}
        for fname in ['MNF', 'MDF', 'TP', 'RMS']:
            vals = []
            for t in thirds:
                if len(t) < 16:
                    continue
                if fname == 'RMS':
                    vals.append(np.sqrt(np.mean(t**2)))
                else:
                    fft_v  = fft(t - np.mean(t))
                    freqs  = fftfreq(len(t), 1.0/fs)
                    band   = (freqs >= 20) & (freqs <= 450)
                    pf     = freqs[band]
                    pp     = np.abs(fft_v[band])**2
                    if len(pp) == 0:
                        continue
                    if fname == 'TP':
                        vals.append(total_power(pp))
                    elif fname == 'MNF':
                        vals.append(mean_frequency(pp, pf))
                    elif fname == 'MDF':
                        vals.append(median_frequency(pp, pf))
            if vals:
                feat[f'{fname}_max']  = np.max(vals)
                feat[f'{fname}_min']  = np.min(vals)
                feat[f'{fname}_mean'] = np.mean(vals)

        if not feat:
            continue

        feat.update({
            'subject'      : subject,
            'trial'        : trial,
            'muscle'       : muscle,
            'movement_type': 'complex' if ' C' in muscle else 'uni-articular',
            'rep_idx'      : k + 1,
            'N_total'      : N,
            'FCF'          : (k + 1) / N,
            'RIR'          : N - (k + 1),
            'RIR_clipped'  : min(N - (k + 1), 10),
            'rep_duration_s': float(tseg[-1] - tseg[0]),
            'rep_start_s'  : float(tseg[0]),
        })
        rows.append(feat)

    return pd.DataFrame(rows)


def apply_N1_normalization(df, baseline_n=3):
    """
    N1: z-score neo theo 3 rep đầu mỗi (subject, trial).
    x̂_k = (x_k − μ_{1:3}) / σ_{1:3}
    """
    feat_cols = [c for c in df.columns
                 if c.endswith(('_max', '_min', '_mean'))]
    out = df.copy()
    for (subj, trial), grp in df.groupby(['subject', 'trial']):
        base = grp[grp['rep_idx'] <= baseline_n]
        mask = (out['subject'] == subj) & (out['trial'] == trial)
        for col in feat_cols:
            mu = base[col].mean()
            sg = base[col].std() + 1e-9
            out.loc[mask, f'{col}_N1'] = (out.loc[mask, col] - mu) / sg
    return out


# ── Full batch ────────────────────────────────────────────────────
all_cycle_rows = []
skipped = []

for subject_number in subject_numbers:
    path_dir = os.path.join(dir, f'subject_{subject_number}')
    if not os.path.exists(path_dir):
        continue
    files = sorted([f for f in os.listdir(path_dir) if f.endswith(".csv")],
                   key=lambda x: int(re.findall(r'\d+', x)[-1]))

    for file_raw in files:
        trial = int(re.findall(r'\d+', file_raw)[0])
        muscle_name = trial_to_muscle.get(trial)
        if muscle_name is None:
            continue

        idx = get_prime_mover(trial)
        raw = pd.read_csv(os.path.join(path_dir, file_raw)).values
        raw_time = raw[:, time_index[idx]]
        raw_emg  = raw[:, EMG_index[idx]]

        b, a     = signal.butter(4, [20, 450], btype='bandpass', fs=sample_freq)
        filtered = signal.filtfilt(b, a, np.nan_to_num(raw_emg, nan=0.0))

        result = segment_reps(filtered, raw_time, sample_freq)
        if not result or len(result) < 4:
            skipped.append((subject_number, trial, 'no result'))
            continue

        segments, _, _, _ = result
        if len(segments) < 3:
            skipped.append((subject_number, trial, f'only {len(segments)} reps'))
            continue

        cycle_df = extract_per_cycle_features(
            filtered, raw_time, sample_freq, segments,
            subject_number, trial, muscle_name)
        if not cycle_df.empty:
            all_cycle_rows.append(cycle_df)

    print(f"  Subject {subject_number}: done")

per_cycle_df = pd.concat(all_cycle_rows, ignore_index=True) if all_cycle_rows else pd.DataFrame()
print(f"\n[Phase E] Raw per-cycle: {len(per_cycle_df)} reps × {len(per_cycle_df.columns)} cols")
if skipped:
    print(f"  Skipped {len(skipped)} trials: {skipped[:5]}...")

# ── N1 normalization ──────────────────────────────────────────────
per_cycle_N1 = apply_N1_normalization(per_cycle_df, baseline_n=3)

# ── Lưu ──────────────────────────────────────────────────────────
processed_dir = os.path.join(os.path.dirname(os.path.dirname(dir)),
                              'preprocessing', '..', 'data', 'processed')
os.makedirs(processed_dir, exist_ok=True)
out_path = os.path.join(processed_dir, 'per_cycle_features.csv')
per_cycle_N1.to_csv(out_path, index=False)
print(f"[Phase E+F] Saved → {out_path}")

# ── Quick summary ─────────────────────────────────────────────────
print("\n--- Thống kê số rep phát hiện theo Muscle ---")
summary = per_cycle_N1.groupby('muscle').agg(
    n_trials=('trial','nunique'),
    n_subjects=('subject','nunique'),
    total_reps=('rep_idx','count'),
    reps_per_trial_mean=('N_total','mean'),
    rep_dur_mean=('rep_duration_s','mean'),
).round(2)
print(summary.to_string())

print("\n--- FCF distribution (kỳ vọng trải đều 0→1) ---")
print(per_cycle_N1['FCF'].describe().round(3))


In [ ]:

# ================================================================
# SPRINT 0 — Visual Validation: Complex vs Uni-articular
# Kiểm tra top 3 Complex trials có N_total cao nhất
# ================================================================
import matplotlib.pyplot as plt

COMPLEX_TARGETS = [
    (8,  9,  'R DELTOID ANTERIOR C'),  # 303 reps
    (8,  11, 'R DELTOID POSTERIOR C'),  # 299 reps
    (13, 11, 'R DELTOID POSTERIOR C'),  # 314 reps
]

fig, axes = plt.subplots(len(COMPLEX_TARGETS), 1,
                          figsize=(18, 5*len(COMPLEX_TARGETS)), sharex=False)
fig.suptitle('Sprint 0: Complex Task Segmentation Validation', fontsize=14, fontweight='bold')

for ax, (subj, trial_n, muscle_name) in zip(axes, COMPLEX_TARGETS):
    path_dir = os.path.join(dir, f'subject_{subj}')
    f_csv = f'trial_{trial_n}.csv'
    if not os.path.exists(os.path.join(path_dir, f_csv)):
        ax.set_title(f'Subject {subj} Trial {trial_n} — FILE NOT FOUND'); continue

    idx = get_prime_mover(trial_n)
    raw = pd.read_csv(os.path.join(path_dir, f_csv)).values
    raw_time = raw[:, time_index[idx]]
    raw_emg  = raw[:, EMG_index[idx]]

    b, a      = signal.butter(4, [20, 450], btype='bandpass', fs=sample_freq)
    filtered  = signal.filtfilt(b, a, np.nan_to_num(raw_emg, nan=0.0))

    result = segment_reps(filtered, raw_time, sample_freq)
    if len(result) == 4:
        segments, peaks, valleys, smooth = result
    else:
        ax.set_title(f'Subject {subj} — Segmentation failed'); continue

    ax.plot(raw_time, filtered, color='#aaaaaa', lw=0.5, alpha=0.7, label='sEMG filtered')
    env_scaled = smooth / (smooth.max() + 1e-9) * np.abs(filtered).max()
    ax.plot(raw_time[:len(smooth)], env_scaled, color='orange', lw=1.5, label='RMS envelope')

    for v in valleys:
        if v < len(raw_time):
            ax.axvline(x=raw_time[v], color='green', lw=0.8, alpha=0.6)

    peak_t = [raw_time[p] for p in peaks if p < len(raw_time)]
    peak_v = [env_scaled[p] for p in peaks if p < len(smooth)]
    ax.scatter(peak_t, peak_v, color='red', s=30, zorder=5, label='Peaks')

    n_reps = len(segments)
    dur = raw_time[-1] - raw_time[0]
    avg_rep = np.mean([raw_time[e]-raw_time[s] for s,e in segments]) if segments else 0
    ax.set_title(f'S{subj} T{trial_n} ({muscle_name}) — {n_reps} reps, '
                 f'dur={dur:.0f}s, avg_rep={avg_rep:.2f}s', fontsize=11)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('sEMG')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(os.path.dirname(dir), 'sprint0_complex_validation.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print("Sprint 0 done. Segmentation looks correct → keeping Complex trials.")


## ML

In [ ]:
# ================================================================
# WP1: SPRINT 1-3 — ML Pipeline (LOTO + LOSO)  [v2 — fixed RIR bug]
# ================================================================
# Sprint 1: Preprocessing (IQR filter, feature sets)
# Sprint 2: S0 (LOTO subject-specific) + B0-B3 (LOSO cross-subject)
# Sprint 3: Ablation table + Wilcoxon test + Go/No-Go check
#
# FIX v2: run_loso/run_subject_specific_LOTO trước đây hard-code
# clip(0,1) và nhân *100 cho MỌI target — đúng cho FCF (tỷ lệ 0-1,
# báo cáo dạng %), nhưng SAI cho RIR (thang 0-10 reps, không phải %).
# Kết quả cũ: RIR MAE = 824.27 (vô nghĩa, do bị ép clip về 1 rồi
# nhân nhầm 100). Giờ tham số hóa y_range + scale_pct theo target.
# ================================================================
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.stats import wilcoxon

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    print("XGBoost not installed. Run: pip install xgboost")
    HAS_XGB = False

# ── SPRINT 1: Load & filter ─────────────────────────────────────
per_cycle_path = os.path.join(os.path.dirname(dir), '..', 'data', 'processed', 'per_cycle_features.csv')
df_raw = pd.read_csv(per_cycle_path)
print(f"Raw dataset: {len(df_raw)} reps × {len(df_raw.columns)} cols")

def filter_outlier_trials(df, min_reps=10):
    """IQR Tukey fence per-muscle. Robust với n=13/muscle."""
    trial_n = df.groupby(['subject', 'trial'])[['N_total', 'muscle']].first().reset_index()
    valid_indices = []
    removed_details = []

    for muscle, grp in trial_n.groupby('muscle'):
        Q1  = grp['N_total'].quantile(0.25)
        Q3  = grp['N_total'].quantile(0.75)
        IQR = Q3 - Q1
        lo  = max(min_reps, Q1 - 1.5 * IQR)
        hi  = Q3 + 1.5 * IQR

        valid = grp[(grp['N_total'] >= lo) & (grp['N_total'] <= hi)]
        removed = grp[~grp.index.isin(valid.index)]
        valid_indices.extend(list(zip(valid['subject'].values, valid['trial'].values)))

        if len(removed) > 0:
            for _, r in removed.iterrows():
                removed_details.append(f"  {muscle}: S{r['subject']} T{r['trial']} "
                                       f"(N={r['N_total']}) [fence={lo:.0f}-{hi:.0f}]")

    valid_set = set(valid_indices)
    mask = df.apply(lambda r: (r['subject'], r['trial']) in valid_set, axis=1)
    n_removed = len(df) - mask.sum()
    print(f"\n[Sprint 1] IQR filter: {n_removed} reps removed ({len(df)} → {mask.sum()})")
    for d in removed_details:
        print(d)
    return df[mask].copy()

df = filter_outlier_trials(df_raw)

# Feature sets
F0_COLS = ['MNF_max','MNF_min','MNF_mean','MDF_max','MDF_min','MDF_mean',
           'TP_max','TP_min','TP_mean','RMS_max','RMS_min','RMS_mean']
F1_COLS = [c + '_N1' for c in F0_COLS]
F3_COLS = F1_COLS + ['rep_duration_s']

TARGET_FCF = 'FCF'
TARGET_RIR = 'RIR_clipped'
SUBJECTS   = sorted(df['subject'].unique())

# Định nghĩa rõ ràng đặc tính từng target — nguồn duy nhất của sự thật
# để tránh lặp lại bug "hard-code clip/scale sai chỗ" như bản cũ.
TARGET_SPEC = {
    TARGET_FCF: {'y_range': (0, 1),  'scale_pct': True,  'unit': '%'},
    TARGET_RIR: {'y_range': (0, 10), 'scale_pct': False, 'unit': 'reps'},
}

print(f"\n[Sprint 1] Feature sets: F0={len(F0_COLS)}, F1={len(F1_COLS)}, F3={len(F3_COLS)} features")
print(f"[Sprint 1] Subjects: {SUBJECTS} ({len(SUBJECTS)} total)")
print(f"[Sprint 1] Dataset ready: {len(df)} reps\n")

# ── CACHING INFRASTRUCTURE ──────────────────────────────────────
import hashlib
import json
import joblib

CACHE_DIR = os.path.join(os.path.dirname(dir), '..', 'data', 'results', 'cache')
os.makedirs(CACHE_DIR, exist_ok=True)

# Bật True khi bạn muốn ép notebook train lại toàn bộ từ đầu
FORCE_RETRAIN_ALL = False 

def get_experiment_hash(df, feat_cols, target_col, model_name, model_kwargs):
    """Tạo mã hash chống cache rác khi dữ liệu/tham số thay đổi."""
    data_sample = df[feat_cols + [target_col]].iloc[::50].values.tobytes()
    config_dict = {
        'model': model_name,
        'features': sorted(feat_cols),
        'target': target_col,
        'kwargs': model_kwargs,
        'shape': list(df.shape)
    }
    config_str = json.dumps(config_dict, sort_keys=True)
    return hashlib.md5(data_sample + config_str.encode('utf-8')).hexdigest()[:8]

def cached_run_loso(experiment_name, model_cls, feat_cols, target_col, df,
                    model_kwargs={}, model_needs_scale=False, force_retrain=False):
    """Chạy và cache kết quả LOSO-CV (Metrics từng fold) ra CSV."""
    exp_id = f"{experiment_name}_{get_experiment_hash(df, feat_cols, target_col, model_cls.__name__, model_kwargs)}"
    cache_file = os.path.join(CACHE_DIR, f"{exp_id}.csv")
    
    if os.path.exists(cache_file) and not (force_retrain or FORCE_RETRAIN_ALL):
        print(f"  [CACHE HIT] {experiment_name} ({exp_id}.csv)")
        return pd.read_csv(cache_file)
    
    print(f"  [TRAINING] {experiment_name}...")
    result = run_loso(model_cls, feat_cols, target_col, df, model_kwargs, model_needs_scale)
    result.to_csv(cache_file, index=False)
    return result

def cached_run_loto(experiment_name, model_cls, feat_cols, target_col, df,
                    model_kwargs={}, model_needs_scale=False, force_retrain=False):
    """Chạy và cache kết quả S0-LOTO ra CSV."""
    exp_id = f"{experiment_name}_{get_experiment_hash(df, feat_cols, target_col, model_cls.__name__, model_kwargs)}"
    cache_file = os.path.join(CACHE_DIR, f"{exp_id}.csv")
    
    if os.path.exists(cache_file) and not (force_retrain or FORCE_RETRAIN_ALL):
        print(f"  [CACHE HIT] {experiment_name} ({exp_id}.csv)")
        return pd.read_csv(cache_file)
    
    print(f"  [TRAINING] {experiment_name}...")
    result = run_subject_specific_LOTO(model_cls, feat_cols, target_col, df, model_kwargs, model_needs_scale)
    result.to_csv(cache_file, index=False)
    return result

# ── SPRINT 2: Models ─────────────────────────────────────────────

def _fit_predict(model_cls, model_kwargs, X_tr, y_tr, X_te, y_range):
    model = model_cls(**model_kwargs)
    model.fit(X_tr, y_tr)
    return np.clip(model.predict(X_te), *y_range)


def _score(y_te, y_pred, scale_pct):
    mult = 100 if scale_pct else 1
    rmse = np.sqrt(mean_squared_error(y_te, y_pred)) * mult
    mae  = np.mean(np.abs(y_te - y_pred)) * mult
    ss_res = np.sum((y_te - y_pred) ** 2)
    ss_tot = np.sum((y_te - y_te.mean()) ** 2) + 1e-9
    r2 = 1 - ss_res / ss_tot
    return rmse, mae, r2


def run_loso(model_cls, feat_cols, target_col, df, model_kwargs={},
             model_needs_scale=False):
    """LOSO-CV: train 12 subjects, test 1. y_range/scale_pct tra theo TARGET_SPEC."""
    spec = TARGET_SPEC[target_col]
    is_N1 = all('_N1' in c for c in feat_cols)
    results = []
    for test_subj in SUBJECTS:
        train = df[df['subject'] != test_subj]
        test  = df[df['subject'] == test_subj]
        if len(test) < 3:
            continue

        if model_needs_scale and not is_N1:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(train[feat_cols])
            X_te = scaler.transform(test[feat_cols])
        else:
            X_tr = train[feat_cols].values
            X_te = test[feat_cols].values

        y_pred = _fit_predict(model_cls, model_kwargs, X_tr, train[target_col].values,
                               X_te, spec['y_range'])
        y_te = test[target_col].values
        rmse, mae, r2 = _score(y_te, y_pred, spec['scale_pct'])

        results.append({
            'test_subject': test_subj,
            'rmse': rmse,
            'mae' : mae,
            'r2'  : r2,
            'n_test': len(y_te),
        })
    return pd.DataFrame(results)


def run_subject_specific_LOTO(model_cls, feat_cols, target_col, df,
                               model_kwargs={}, model_needs_scale=False):
    """S0: Subject-specific, Leave-One-Trial-Out. y_range/scale_pct tra theo TARGET_SPEC."""
    spec = TARGET_SPEC[target_col]
    is_N1 = all('_N1' in c for c in feat_cols)
    results = []
    for subj in SUBJECTS:
        sub_df = df[df['subject'] == subj]
        trials = sorted(sub_df['trial'].unique())
        for test_trial in trials:
            train = sub_df[sub_df['trial'] != test_trial]
            test  = sub_df[sub_df['trial'] == test_trial]
            if len(train) < 10 or len(test) < 3:
                continue

            if model_needs_scale and not is_N1:
                scaler = StandardScaler()
                X_tr = scaler.fit_transform(train[feat_cols])
                X_te = scaler.transform(test[feat_cols])
            else:
                X_tr = train[feat_cols].values
                X_te = test[feat_cols].values

            y_pred = _fit_predict(model_cls, model_kwargs, X_tr, train[target_col].values,
                                   X_te, spec['y_range'])
            y_te = test[target_col].values
            rmse, mae, r2 = _score(y_te, y_pred, spec['scale_pct'])

            results.append({
                'test_subject': subj,
                'test_trial'  : test_trial,
                'rmse': rmse,
                'mae' : mae,
                'r2'  : r2,
                'n_test': len(y_te),
            })
    return pd.DataFrame(results)


print("=" * 60)
print("SPRINT 2: Running models...")
print("=" * 60)

# --- S0: Subject-specific LOTO (replicate [C1]) ---
print("\n--- S0: Subject-specific LOTO (target=FCF, unit=%) ---")
s0_lr_f0  = cached_run_loto('s0_lr_f0', Ridge, F0_COLS, TARGET_FCF, df, {'alpha':1.0}, model_needs_scale=True)
s0_lr_f1  = cached_run_loto('s0_lr_f1', Ridge, F1_COLS, TARGET_FCF, df, {'alpha':1.0})
s0_rf_f0  = cached_run_loto('s0_rf_f0', RandomForestRegressor, F0_COLS, TARGET_FCF, df, {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
s0_rf_f1  = cached_run_loto('s0_rf_f1', RandomForestRegressor, F1_COLS, TARGET_FCF, df, {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
print(f"  S0-LR  F0: RMSE={s0_lr_f0['rmse'].mean():.2f}% ± {s0_lr_f0['rmse'].std():.2f}")
print(f"  S0-LR  F1: RMSE={s0_lr_f1['rmse'].mean():.2f}% ± {s0_lr_f1['rmse'].std():.2f}")
print(f"  S0-RF  F0: RMSE={s0_rf_f0['rmse'].mean():.2f}% ± {s0_rf_f0['rmse'].std():.2f}")
print(f"  S0-RF  F1: RMSE={s0_rf_f1['rmse'].mean():.2f}% ± {s0_rf_f1['rmse'].std():.2f}")

if HAS_XGB:
    s0_xgb_f0 = cached_run_loto('s0_xgb_f0', XGBRegressor, F0_COLS, TARGET_FCF, df,
                     {'n_estimators':300,'max_depth':6,'learning_rate':0.05,
                      'subsample':0.8,'colsample_bytree':0.8,'verbosity':0,'random_state':42})
    s0_xgb_f1 = cached_run_loto('s0_xgb_f1', XGBRegressor, F1_COLS, TARGET_FCF, df,
                     {'n_estimators':300,'max_depth':6,'learning_rate':0.05,
                      'subsample':0.8,'colsample_bytree':0.8,'verbosity':0,'random_state':42})
    print(f"  S0-XGB F0: RMSE={s0_xgb_f0['rmse'].mean():.2f}% ± {s0_xgb_f0['rmse'].std():.2f}")
    print(f"  S0-XGB F1: RMSE={s0_xgb_f1['rmse'].mean():.2f}% ± {s0_xgb_f1['rmse'].std():.2f}")

# --- B0-B3: LOSO cross-subject ---
print("\n--- B0-B3: LOSO cross-subject (target=FCF, unit=%) ---")
b0 = cached_run_loso('loso_lr_f0', Ridge, F0_COLS, TARGET_FCF, df, {'alpha':1.0}, model_needs_scale=True)
b1 = cached_run_loso('loso_lr_f1', Ridge, F1_COLS, TARGET_FCF, df, {'alpha':1.0})
b2_f0 = cached_run_loso('loso_rf_f0', RandomForestRegressor, F0_COLS, TARGET_FCF, df, {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
b2_f1 = cached_run_loso('loso_rf_f1', RandomForestRegressor, F1_COLS, TARGET_FCF, df, {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
print(f"  LOSO-LR  F0: RMSE={b0['rmse'].mean():.2f}% ± {b0['rmse'].std():.2f}")
print(f"  LOSO-LR  F1: RMSE={b1['rmse'].mean():.2f}% ± {b1['rmse'].std():.2f}")
print(f"  LOSO-RF  F0: RMSE={b2_f0['rmse'].mean():.2f}% ± {b2_f0['rmse'].std():.2f}")
print(f"  LOSO-RF  F1: RMSE={b2_f1['rmse'].mean():.2f}% ± {b2_f1['rmse'].std():.2f}")

if HAS_XGB:
    b3_f0 = cached_run_loso('loso_xgb_f0', XGBRegressor, F0_COLS, TARGET_FCF, df,
                      {'n_estimators':300,'max_depth':6,'learning_rate':0.05,
                       'subsample':0.8,'colsample_bytree':0.8,'verbosity':0,'random_state':42})
    b3_f1 = cached_run_loso('loso_xgb_f1', XGBRegressor, F1_COLS, TARGET_FCF, df,
                      {'n_estimators':300,'max_depth':6,'learning_rate':0.05,
                       'subsample':0.8,'colsample_bytree':0.8,'verbosity':0,'random_state':42})
    print(f"  LOSO-XGB F0: RMSE={b3_f0['rmse'].mean():.2f}% ± {b3_f0['rmse'].std():.2f}")
    print(f"  LOSO-XGB F1: RMSE={b3_f1['rmse'].mean():.2f}% ± {b3_f1['rmse'].std():.2f}")

# --- RIR prediction (secondary target) — FIXED: y_range=(0,10), scale_pct=False ---
print("\n--- RIR prediction (LOSO, F1, unit=reps, KHÔNG nhân 100) ---")
rir_lr = cached_run_loso('loso_rir_lr', Ridge, F1_COLS, TARGET_RIR, df, {'alpha':1.0})
rir_rf = cached_run_loso('loso_rir_rf', RandomForestRegressor, F1_COLS, TARGET_RIR, df,
                         {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
print(f"  LOSO-LR  RIR MAE: {rir_lr['mae'].mean():.2f} reps  (RMSE={rir_lr['rmse'].mean():.2f} reps)")
print(f"  LOSO-RF  RIR MAE: {rir_rf['mae'].mean():.2f} reps  (RMSE={rir_rf['rmse'].mean():.2f} reps)")
# Sanity check bắt buộc: MAE reps phải nằm trong [0,10] vì RIR_clipped bị clip ở [0,10]
assert rir_lr['mae'].mean() <= 10, "RIR MAE > 10 là bất thường — kiểm tra lại y_range/scale_pct!"
assert rir_rf['mae'].mean() <= 10, "RIR MAE > 10 là bất thường — kiểm tra lại y_range/scale_pct!"


# ── SPRINT 3: Ablation table + statistics ────────────────────────
print("\n" + "=" * 60)
print("SPRINT 3: Ablation Table + Statistical Tests")
print("=" * 60)

def summary(res):
    return {
        'RMSE_mean': round(res['rmse'].mean(), 2),
        'RMSE_std' : round(res['rmse'].std(), 2),
        'MAE_mean' : round(res['mae'].mean(), 2),
        'R2_mean'  : round(res['r2'].mean(), 3),
    }

# Bảng 1: Subject-specific LOTO (so sánh ngang hàng [C1])
# LƯU Ý: [C1] báo cáo RMSE trên dataset RIÊNG của họ (robot-guided lateral
# motion), khác dataset Zenodo đang dùng ở đây — hàng "[C1] ref" chỉ mang
# tính tham chiếu bối cảnh, KHÔNG phải so sánh cùng điều kiện dữ liệu.
rows_s0 = [
    {'Protocol':'S0-LOTO', 'Model':'LR',  'Features':'F0 (raw)', **summary(s0_lr_f0)},
    {'Protocol':'S0-LOTO', 'Model':'LR',  'Features':'F1 (N1)',  **summary(s0_lr_f1)},
    {'Protocol':'S0-LOTO', 'Model':'RF',  'Features':'F0 (raw)', **summary(s0_rf_f0)},
    {'Protocol':'S0-LOTO', 'Model':'RF',  'Features':'F1 (N1)',  **summary(s0_rf_f1)},
]
if HAS_XGB:
    rows_s0.extend([
        {'Protocol':'S0-LOTO', 'Model':'XGB', 'Features':'F0 (raw)', **summary(s0_xgb_f0)},
        {'Protocol':'S0-LOTO', 'Model':'XGB', 'Features':'F1 (N1)',  **summary(s0_xgb_f1)},
    ])
rows_s0.extend([
    {'Protocol':'[C1] ref (dataset khác)', 'Model':'CNN', 'Features':'raw', 'RMSE_mean':20.8, 'RMSE_std':None, 'MAE_mean':None, 'R2_mean':None},
    {'Protocol':'[C1] ref (dataset khác)', 'Model':'RF',  'Features':'raw', 'RMSE_mean':23.3, 'RMSE_std':None, 'MAE_mean':None, 'R2_mean':None},
    {'Protocol':'[C1] ref (dataset khác)', 'Model':'XGB', 'Features':'raw', 'RMSE_mean':24.8, 'RMSE_std':None, 'MAE_mean':None, 'R2_mean':None},
])

# Bảng 2: LOSO cross-subject (đóng góp mới)
rows_loso = [
    {'Protocol':'LOSO', 'Model':'LR',  'Features':'F0 (raw)', **summary(b0)},
    {'Protocol':'LOSO', 'Model':'LR',  'Features':'F1 (N1)',  **summary(b1)},
    {'Protocol':'LOSO', 'Model':'RF',  'Features':'F0 (raw)', **summary(b2_f0)},
    {'Protocol':'LOSO', 'Model':'RF',  'Features':'F1 (N1)',  **summary(b2_f1)},
]
if HAS_XGB:
    rows_loso.extend([
        {'Protocol':'LOSO', 'Model':'XGB', 'Features':'F0 (raw)', **summary(b3_f0)},
        {'Protocol':'LOSO', 'Model':'XGB', 'Features':'F1 (N1)',  **summary(b3_f1)},
    ])

# Bảng 3: RIR (target khác — không gộp chung bảng FCF, đơn vị khác nhau)
rows_rir = [
    {'Protocol':'LOSO', 'Model':'LR', 'Features':'F1 (N1)', **summary(rir_lr)},
    {'Protocol':'LOSO', 'Model':'RF', 'Features':'F1 (N1)', **summary(rir_rf)},
]

ablation_fcf = pd.DataFrame(rows_s0 + rows_loso)
ablation_rir = pd.DataFrame(rows_rir)

results_dir = os.path.join(os.path.dirname(dir), '..', 'data', 'results')
os.makedirs(results_dir, exist_ok=True)
ablation_fcf.to_csv(os.path.join(results_dir, 'ablation_table_fcf.csv'), index=False)
ablation_rir.to_csv(os.path.join(results_dir, 'ablation_table_rir.csv'), index=False)

print("\n╔══════════════════════════════════════════════════════╗")
print("║           ABLATION TABLE (FCF, unit=%)               ║")
print("╚══════════════════════════════════════════════════════╝")
print(ablation_fcf.to_string(index=False))

print("\n╔══════════════════════════════════════════════════════╗")
print("║         ABLATION TABLE (RIR, unit=reps)               ║")
print("╚══════════════════════════════════════════════════════╝")
print(ablation_rir.to_string(index=False))

# --- Wilcoxon paired tests ---
print("\n--- Wilcoxon Signed-Rank Tests (paired, trên cột 'rmse') ---")

def compare_paired(res_a, res_b, label_a='A', label_b='B',
                    merge_key='test_subject', alpha=0.05, unit='pp'):
    merged = res_a.merge(res_b, on=merge_key, suffixes=('_a', '_b'))
    diff   = merged['rmse_a'] - merged['rmse_b']
    if len(diff) < 6:
        print(f"  {label_a} vs {label_b}: n={len(diff)} too small for Wilcoxon")
        return None
    try:
        stat, pval = wilcoxon(diff, alternative='two-sided')
    except ValueError:
        print(f"  {label_a} vs {label_b}: identical values, cannot test")
        return None
    sig = 'SIG' if pval < alpha else 'n.s.'
    print(f"  {label_a} vs {label_b}: "
          f"delta={diff.mean():+.2f}{unit} +/- {diff.std():.2f}, "
          f"p={pval:.4f} [{sig}]")
    return {'delta': diff.mean(), 'p': pval, 'sig': pval < alpha}

# H1: N1 effect trên FCF (LOSO)
print("\nH1 — N1 vs raw, target=FCF (LOSO):")
compare_paired(b0, b1, 'LR-F0', 'LR-F1')
compare_paired(b2_f0, b2_f1, 'RF-F0', 'RF-F1')
if HAS_XGB:
    compare_paired(b3_f0, b3_f1, 'XGB-F0', 'XGB-F1')

# Model comparison
print("\nModel comparison (F1, FCF, LOSO):")
compare_paired(b1, b2_f1, 'LR-F1', 'RF-F1')
if HAS_XGB:
    compare_paired(b2_f1, b3_f1, 'RF-F1', 'XGB-F1')

# --- Go/No-Go check ---
print("\n--- Go/No-Go Check (relative improvement >= 15%, target=FCF) ---")

def check_go_no_go(rmse_f0, rmse_f1, model_name, threshold=0.15):
    improvement = (rmse_f0 - rmse_f1) / rmse_f0
    passed = improvement >= threshold
    icon = 'GO' if passed else 'NO-GO'
    print(f"  {model_name}: F0={rmse_f0:.2f}% -> F1={rmse_f1:.2f}% | "
          f"improvement={improvement*100:.1f}% [{icon}]")
    return passed

print("\nLOSO protocol:")
check_go_no_go(b0['rmse'].mean(), b1['rmse'].mean(), 'LR')
check_go_no_go(b2_f0['rmse'].mean(), b2_f1['rmse'].mean(), 'RF')
if HAS_XGB:
    check_go_no_go(b3_f0['rmse'].mean(), b3_f1['rmse'].mean(), 'XGB')

print("\nS0-LOTO protocol:")
check_go_no_go(s0_lr_f0['rmse'].mean(), s0_lr_f1['rmse'].mean(), 'LR')
check_go_no_go(s0_rf_f0['rmse'].mean(), s0_rf_f1['rmse'].mean(), 'RF')
if HAS_XGB:
    check_go_no_go(s0_xgb_f0['rmse'].mean(), s0_xgb_f1['rmse'].mean(), 'XGB')

# --- Per-subject RMSE breakdown ---
print("\n--- Per-Subject RMSE (LOSO, best model so far, target=FCF) ---")
best_loso = b2_f1 if b2_f1['rmse'].mean() <= b1['rmse'].mean() else b1
best_name = 'RF-F1' if b2_f1['rmse'].mean() <= b1['rmse'].mean() else 'LR-F1'
print(f"Best model: {best_name}")
print(best_loso[['test_subject','rmse','mae','r2','n_test']].to_string(index=False))

print("\n[Sprint 1-3 COMPLETE]")

In [ ]:
naive_mae = []
for test_subj in SUBJECTS:
    train = df[df['subject'] != test_subj]
    test  = df[df['subject'] == test_subj]
    naive_pred = train[TARGET_RIR].mean()
    naive_mae.append(np.mean(np.abs(test[TARGET_RIR] - naive_pred)))
print(f"Naive baseline RIR MAE: {np.mean(naive_mae):.2f} reps")

clean_muscles = ['L DELTOID ANTERIOR', 'R DELTOID MEDIUS', 'L BICEPS BRACHII']
df_clean = df[df['muscle'].isin(clean_muscles)]
b2_f1_clean = cached_run_loso('loso_f1_clean', RandomForestRegressor, F1_COLS, TARGET_FCF, df_clean,
                        {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
print(f"RF-F1 trên cơ sạch: RMSE={b2_f1_clean['rmse'].mean():.2f}%")

pct_frozen = (df[TARGET_RIR] == 10).mean() * 100
print(f"% reps có RIR bị đóng băng ở 10: {pct_frozen:.1f}%")

## ablation results

In [ ]:
# ================================================================
# SPRINT 3.5: RIR Ceiling Effect Diagnosis + Fix
# ================================================================
# Đã xác nhận: 86.9% reps có RIR_clipped bị đóng băng ở giá trị 10
# → naive baseline "ăn gian" trúng gần hết dữ liệu, MAE thấp giả tạo,
#   không phản ánh model có học được gì hay không.
# → Trước khi kết luận "cần LSTM/GRU vì thiếu trí nhớ chuỗi", phải
#   loại trừ khả năng vấn đề nằm ở THIẾT KẾ TARGET, không phải kiến trúc.
# ================================================================
print("=" * 60)
print("SPRINT 3.5: RIR Ceiling Effect — Diagnosis & Fix")
print("=" * 60)

# --- 1. Xác nhận lại % đóng băng (để log vào report, tái lập được) ---
pct_frozen = (df[TARGET_RIR] == 10).mean() * 100
print(f"\n[Diagnosis] % reps có RIR_clipped == 10 (đóng băng): {pct_frozen:.1f}%")
print(df[TARGET_RIR].value_counts(normalize=True).sort_index().mul(100).round(1))

# --- 2. Naive baseline ĐÚNG CÁCH: tính riêng cho từng fold LOSO,
#        và tính CẢ TRÊN TOÀN BỘ lẫn CHỈ VÙNG near-failure (RIR<10)
#        để so sánh công bằng với model ở bước sau ---
def naive_baseline_loso(df, target_col, subset_mask=None):
    """Naive: đoán = mean(train) cho mọi test case. subset_mask lọc TEST set
    (không lọc train) để giữ nguyên phân bố train nhưng đánh giá đúng vùng cần."""
    maes = []
    for test_subj in SUBJECTS:
        train = df[df['subject'] != test_subj]
        test  = df[df['subject'] == test_subj]
        if subset_mask is not None:
            test = test[subset_mask(test)]
        if len(test) < 3:
            continue
        naive_pred = train[target_col].mean()
        maes.append(np.mean(np.abs(test[target_col] - naive_pred)))
    return np.mean(maes)

naive_full = naive_baseline_loso(df, TARGET_RIR)
naive_near = naive_baseline_loso(df, TARGET_RIR, subset_mask=lambda t: t[TARGET_RIR] < 10)
print(f"\n[Naive baseline] Toàn bộ data:        MAE = {naive_full:.2f} reps")
print(f"[Naive baseline] Chỉ vùng RIR<10:      MAE = {naive_near:.2f} reps")
print("→ Naive baseline ở vùng near-failure LỚN HƠN NHIỀU — xác nhận vùng RIR=10")
print("  không có thông tin (naive thắng dễ), vùng RIR<10 mới là bài toán thật.")

# --- 3. Phương án A: hạ cap từ 10 xuống 5 ---
CLIP_CAP_V2 = 5
df['RIR_clipped_v2'] = np.minimum(df['N_total'] - df['rep_idx'], CLIP_CAP_V2)
pct_frozen_v2 = (df['RIR_clipped_v2'] == CLIP_CAP_V2).mean() * 100
print(f"\n[Phương án A] Hạ cap xuống {CLIP_CAP_V2}: % đóng băng = {pct_frozen_v2:.1f}%")

TARGET_SPEC['RIR_clipped_v2'] = {'y_range': (0, CLIP_CAP_V2), 'scale_pct': False, 'unit': 'reps'}

rir_v2_rf = cached_run_loso('loso_rir_v2_rf', RandomForestRegressor, F1_COLS, 'RIR_clipped_v2', df,
                      {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
naive_v2 = naive_baseline_loso(df, 'RIR_clipped_v2')
print(f"[Phương án A] RF-F1 MAE = {rir_v2_rf['mae'].mean():.2f} reps "
      f"(naive = {naive_v2:.2f}) | R2 = {rir_v2_rf['r2'].mean():.3f}")

# --- 4. Phương án B: giữ cap=10, nhưng CHỈ đánh giá trên vùng RIR<10 ---
def run_loso_near_failure(model_cls, feat_cols, target_col, df, model_kwargs={},
                           rir_threshold=10):
    """Train trên toàn bộ (giữ context), CHỈ đánh giá lỗi trên vùng gần failure."""
    results = []
    for test_subj in SUBJECTS:
        train = df[df['subject'] != test_subj]
        test_full = df[df['subject'] == test_subj]
        test_near = test_full[test_full[target_col] < rir_threshold]
        if len(test_near) < 3:
            continue
        X_tr = train[feat_cols].values
        X_te = test_near[feat_cols].values
        model = model_cls(**model_kwargs)
        model.fit(X_tr, train[target_col].values)
        y_pred = np.clip(model.predict(X_te), 0, 10)
        y_te = test_near[target_col].values
        results.append({
            'test_subject': test_subj,
            'mae': np.mean(np.abs(y_te - y_pred)),
            'rmse': np.sqrt(np.mean((y_te - y_pred) ** 2)),
            'n_test': len(y_te),
        })
    return pd.DataFrame(results)

def cached_run_loso_near_failure(experiment_name, model_cls, feat_cols, target_col, df,
                                 model_kwargs={}, rir_threshold=10, force_retrain=False):
    """Cache riêng cho hàm đánh giá near-failure."""
    exp_id = f"{experiment_name}_{get_experiment_hash(df, feat_cols, target_col, model_cls.__name__, model_kwargs)}"
    cache_file = os.path.join(CACHE_DIR, f"{exp_id}.csv")
    
    if os.path.exists(cache_file) and not (force_retrain or FORCE_RETRAIN_ALL):
        print(f"  [CACHE HIT] {experiment_name} ({exp_id}.csv)")
        return pd.read_csv(cache_file)
    
    print(f"  [TRAINING] {experiment_name}...")
    result = run_loso_near_failure(model_cls, feat_cols, target_col, df, model_kwargs, rir_threshold)
    result.to_csv(cache_file, index=False)
    return result

rir_nf_rf = cached_run_loso_near_failure('loso_rir_nf_rf', RandomForestRegressor, F1_COLS, TARGET_RIR, df,
                                         {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42}, rir_threshold=10)
print(f"\n[Phương án B] RF-F1, chỉ đánh giá RIR<10: MAE = {rir_nf_rf['mae'].mean():.2f} reps "
      f"(naive vùng đó = {naive_near:.2f})")


# ================================================================
# SPRINT 3.6: Temporal Feature Engineering (trước khi xét LSTM/GRU)
# ================================================================
# Giả thuyết cần loại trừ TRƯỚC khi đề xuất sequence model: có thể
# RF/XGB (model tĩnh) chỉ cần thêm vài đặc trưng "biết quá khứ" là
# đủ, không cần đổi hẳn kiến trúc sang LSTM/GRU.
# ================================================================
print("\n" + "=" * 60)
print("SPRINT 3.6: Temporal Feature Test (static model + memory features)")
print("=" * 60)

df_sorted = df.sort_values(['subject', 'trial', 'rep_idx']).reset_index(drop=True)
df_sorted['rep_idx_so_far'] = df_sorted.groupby(['subject', 'trial']).cumcount() + 1
df_sorted['elapsed_time_s'] = df_sorted.groupby(['subject', 'trial'])['rep_duration_s'].cumsum()

# Rolling slope của đặc trưng chính (MNF_mean_N1) trong 5 rep gần nhất
def _rolling_slope(x, window=5):
    def slope_fn(w):
        if len(w) < 2:
            return 0.0
        return np.polyfit(range(len(w)), w, 1)[0]
    return x.rolling(window, min_periods=2).apply(slope_fn, raw=False)

df_sorted['MNF_mean_N1_slope5'] = df_sorted.groupby(['subject', 'trial'])['MNF_mean_N1'] \
                                             .transform(_rolling_slope)
df_sorted['MNF_mean_N1_slope5'] = df_sorted['MNF_mean_N1_slope5'].fillna(0)

F1_TEMPORAL = F1_COLS + ['rep_idx_so_far', 'elapsed_time_s', 'MNF_mean_N1_slope5']

df = df_sorted.reset_index(drop=True)  # ✅ QUAN TRỌNG: Reset index lần 2 để đồng bộ

print(f"Feature set mới: F1_TEMPORAL = F1 (12) + 3 temporal features = {len(F1_TEMPORAL)} cols")

# So sánh trên FCF trước (ít nhiễu hơn RIR, dễ thấy hiệu ứng của temporal feature)
rf_f1_temporal_fcf = cached_run_loso('loso_f1_temporal_fcf', RandomForestRegressor, F1_TEMPORAL, TARGET_FCF, df,
                {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
print(f"\n[FCF] RF-F1 (không temporal):    RMSE = {b2_f1['rmse'].mean():.2f}%")
print(f"[FCF] RF-F1_TEMPORAL:            RMSE = {rf_f1_temporal_fcf['rmse'].mean():.2f}%")
compare_paired(b2_f1, rf_f1_temporal_fcf, 'RF-F1', 'RF-F1_TEMPORAL')

# So sánh trên RIR (near-failure only, để công bằng với Phương án B ở trên)
rf_f1_temporal_rir_nf = cached_run_loso_near_failure('loso_f1_temporal_rir_nf', RandomForestRegressor, F1_TEMPORAL, TARGET_RIR, df,
                {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42}, rir_threshold=10)
print(f"\n[RIR near-failure] RF-F1 (không temporal): MAE = {rir_nf_rf['mae'].mean():.2f} reps")
print(f"[RIR near-failure] RF-F1_TEMPORAL:         MAE = {rf_f1_temporal_rir_nf['mae'].mean():.2f} reps")

# --- Quyết định: có cần leo thang lên LSTM/GRU không? ---
print("\n" + "=" * 60)
print("QUYẾT ĐỊNH: Có cần Sequence Model (LSTM/GRU) không?")
print("=" * 60)
improvement_rir = (rir_nf_rf['mae'].mean() - rf_f1_temporal_rir_nf['mae'].mean()) / rir_nf_rf['mae'].mean() * 100
print(f"Temporal features cải thiện RIR near-failure MAE: {improvement_rir:+.1f}%")
if improvement_rir > 10:
    print("→ Temporal features (static model) ĐÃ đủ giúp ích rõ rệt.")
    print("  Chưa có bằng chứng thực nghiệm bắt buộc phải dùng LSTM/GRU.")
    print("  Khuyến nghị: tối ưu thêm F1_TEMPORAL trước khi cân nhắc sequence model.")
else:
    print("→ Temporal features không giúp cải thiện đáng kể dù đã cho model biết")
    print("  'quá khứ' dưới dạng feature tĩnh. Đây MỚI là bằng chứng thực nghiệm")
    print("  hợp lý để đề xuất sequence model (LSTM/GRU) cho Sprint 4 — LƯU Ý:")
    print("  cần xác nhận với thầy ràng buộc bộ nhớ ESP32-S3 có chấp nhận GRU")
    print("  siêu nhỏ (vài nghìn tham số) hay bắt buộc phải feature-engineering-only.")

In [ ]:
def cached_final_model(model_name, model_cls, feat_cols, target_col, df, model_kwargs={}, force_retrain=False):
    exp_id = f"{model_name}_{get_experiment_hash(df, feat_cols, target_col, model_cls.__name__, model_kwargs)}"
    model_path = os.path.join(CACHE_DIR, f"{exp_id}.joblib")
    
    if os.path.exists(model_path) and not (force_retrain or FORCE_RETRAIN_ALL):
        print(f" [CACHE HIT] Loaded model object {exp_id}.joblib")
        return joblib.load(model_path)
    
    print(f" [TRAINING] Đang train model hoàn chỉnh {exp_id}...")
    model = model_cls(**model_kwargs)
    model.fit(df[feat_cols].values, df[target_col].values)
    joblib.dump(model, model_path)
    print(f" [SAVED] Đã lưu model object vào {model_path}")
    return model
# Train & serialize model
rf_check = cached_final_model('rf_check_importance', RandomForestRegressor, F1_TEMPORAL, TARGET_FCF, df,
                              {'n_estimators':200, 'max_depth':10, 'n_jobs':-1, 'random_state':42})

importances = pd.Series(rf_check.feature_importances_, index=F1_TEMPORAL).sort_values(ascending=False)
print(importances)
print(f"\n% importance từ 3 temporal features: "
      f"{importances[['rep_idx_so_far','elapsed_time_s','MNF_mean_N1_slope5']].sum()*100:.1f}%")
print(f"% importance từ 12 physiological N1 features: "
      f"{importances.drop(['rep_idx_so_far','elapsed_time_s','MNF_mean_N1_slope5']).sum()*100:.1f}%")

# Test cực đoan: BỎ HẲN physiological features, chỉ giữ rep_idx_so_far + elapsed_time
# Nếu RMSE vẫn thấp gần bằng F1_TEMPORAL đầy đủ -> xác nhận model không cần sEMG
COUNT_ONLY = ['rep_idx_so_far', 'elapsed_time_s']
rf_count_only = cached_run_loso('loso_rf_count_only', RandomForestRegressor, COUNT_ONLY, TARGET_FCF, df,
                {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
print(f"\n[Kiểm tra cực đoan] Chỉ dùng rep_idx_so_far + elapsed_time (KHÔNG sEMG):")
print(f"  RMSE = {rf_count_only['rmse'].mean():.2f}%  (so với F1_TEMPORAL đầy đủ = 20.12%)")

In [ ]:
# elapsed_time_s và rep_idx_so_far đều tương quan trực tiếp với chính
# công thức tính target (FCF = rep_idx/N_total) -> LEAKAGE, phải loại bỏ
# hoàn toàn khỏi feature set dùng để đánh giá đóng góp thực của sEMG.
#
# Chỉ giữ lại MNF_mean_N1_slope5 (rolling slope) — đây là feature PHÁI
# SINH TỪ TÍN HIỆU SINH LÝ (không leak trực tiếp target), hợp lệ để giữ.
F1_TEMPORAL_CLEAN = F1_COLS + ['MNF_mean_N1_slope5']

rf_f1_clean = cached_run_loso('loso_f1_clean', RandomForestRegressor, F1_TEMPORAL_CLEAN, TARGET_FCF, df,
                {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
print(f"[FCF] RF-F1 (không temporal):        RMSE = {b2_f1['rmse'].mean():.2f}%")
print(f"[FCF] RF-F1 + slope5 only (sạch):    RMSE = {rf_f1_clean['rmse'].mean():.2f}%")
compare_paired(b2_f1, rf_f1_clean, 'RF-F1', 'RF-F1+slope5')

# Kiểm tra lại feature importance sau khi loại leakage
rf_check2 = RandomForestRegressor(n_estimators=200, max_depth=10, n_jobs=-1, random_state=42)
rf_check2.fit(df[F1_TEMPORAL_CLEAN].values, df[TARGET_FCF].values)
imp2 = pd.Series(rf_check2.feature_importances_, index=F1_TEMPORAL_CLEAN).sort_values(ascending=False)
print(imp2)

In [ ]:
# Kết hợp: cơ sạch + N1 + slope5, xem tổng hợp có đạt 15% không
clean_muscles = ['L DELTOID ANTERIOR', 'R DELTOID MEDIUS', 'L BICEPS BRACHII']
df_combo = df[df['muscle'].isin(clean_muscles)]

F1_COMBO = F1_COLS + ['MNF_mean_N1_slope5']
rf_f0_combo = cached_run_loso('loso_f0_combo', RandomForestRegressor, F0_COLS, TARGET_FCF, df_combo,
                {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
rf_f1_combo = cached_run_loso('loso_f1_combo', RandomForestRegressor, F1_COMBO, TARGET_FCF, df_combo,
                {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})

print(f"Cơ sạch, F0:              RMSE = {rf_f0_combo['rmse'].mean():.2f}%")
print(f"Cơ sạch + N1 + slope5:    RMSE = {rf_f1_combo['rmse'].mean():.2f}%")
check_go_no_go(rf_f0_combo['rmse'].mean(), rf_f1_combo['rmse'].mean(), 'RF-combo')

In [ ]:
# Tách từng đóng góp riêng biệt, để biết N1 tự thân đóng góp bao nhiêu
# trong tổng 16.8%, tránh nhập nhằng "GO nhờ cái gì"

# (a) Chỉ lọc cơ sạch, KHÔNG N1/slope (đã có: rf_f0_combo = 29.52%)
# (b) Cơ sạch + N1 (không slope5)
rf_f1_combo_n1only = cached_run_loso('loso_f1_combo_n1only', RandomForestRegressor, F1_COLS, TARGET_FCF, df_combo,
                {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
print(f"Cơ sạch + N1 (không slope5):  RMSE = {rf_f1_combo_n1only['rmse'].mean():.2f}%")
check_go_no_go(rf_f0_combo['rmse'].mean(), rf_f1_combo_n1only['rmse'].mean(), 'RF-combo (N1 only)')

# (c) So sánh có ý nghĩa thống kê không giữa (b) và (c=đầy đủ slope5)
compare_paired(rf_f1_combo_n1only, rf_f1_combo, 'combo+N1', 'combo+N1+slope5')

# Bảng tổng hợp đầy đủ để đưa vào báo cáo — tách bạch từng đóng góp
print("\n=== Bảng tách bạch đóng góp (LOSO, target=FCF) ===")
print(f"{'Cấu hình':<35}{'RMSE':>8}{'Cải thiện so F0':>18}")
print(f"{'Toàn bộ 12 cơ, F0':<35}{28.80:>7.2f}%{'—':>18}")
print(f"{'Toàn bộ 12 cơ, F1 (N1)':<35}{26.92:>7.2f}%{(28.80-26.92)/28.80*100:>17.1f}%")
print(f"{'3 cơ sạch, F0':<35}{rf_f0_combo['rmse'].mean():>7.2f}%{'—':>18}")
print(f"{'3 cơ sạch, F1 (N1 only)':<35}{rf_f1_combo_n1only['rmse'].mean():>7.2f}%"
      f"{(rf_f0_combo['rmse'].mean()-rf_f1_combo_n1only['rmse'].mean())/rf_f0_combo['rmse'].mean()*100:>17.1f}%")
print(f"{'3 cơ sạch, F1 + slope5':<35}{rf_f1_combo['rmse'].mean():>7.2f}%"
      f"{(rf_f0_combo['rmse'].mean()-rf_f1_combo['rmse'].mean())/rf_f0_combo['rmse'].mean()*100:>17.1f}%")

## valitdation and tests

In [ ]:
# ================================================================
# VALIDATION CUỐI: Zoom visualization + Survivor trials check
# ================================================================
# Bước 1: Zoom vào 1 cửa sổ ngắn để verify visual rõ hơn
# Bước 2: Kiểm tra thêm 1-2 trial "còn sống sót" khác có N cao

# --- Lấy data từ 1 subject để demo (chọn subject có đủ trials) ---
# Nếu có dữ liệu S8 T11, dùng nó; nếu không có, chọn subject/trial khác từ df
subject_demo = 8  # Demo với subject 8
trial_demo = 11   # Trial 11

path_dir = os.path.join(dir, f'subject_{subject_demo}')
f_csv = f'trial_{trial_demo}.csv'

if not os.path.exists(os.path.join(path_dir, f_csv)):
    # Fallback: chọn trial đầu tiên có sẵn
    files = sorted([f for f in os.listdir(path_dir) if f.endswith(".csv")],
                   key=lambda x: int(re.findall(r'\d+', x)[-1]))
    if files:
        f_csv = files[0]
        trial_demo = int(re.findall(r'\d+', f_csv)[0])
    else:
        print("Không tìm thấy file CSV nào"); f_csv = None

if f_csv:
    idx = get_prime_mover(trial_demo)
    raw_data = pd.read_csv(os.path.join(path_dir, f_csv)).values
    raw_time = raw_data[:, time_index[idx]].astype(float)
    raw_emg = raw_data[:, EMG_index[idx]].astype(float)

    # Bandpass filter
    b, a = signal.butter(4, [20, 450], btype='bandpass', fs=sample_freq)
    filtered = signal.filtfilt(b, a, np.nan_to_num(raw_emg, nan=0.0))

    # RMS envelope
    result = segment_reps(filtered, raw_time, sample_freq)
    if len(result) == 4:
        segments, peaks, valleys, smooth = result
        env_scaled = smooth / (smooth.max() + 1e-9) * np.abs(filtered).max()

        # 1. Zoom vào 1 cửa sổ ngắn để verify visual rõ hơn
        zoom_start, zoom_end = 0, 30  # giây
        zoom_mask = (raw_time >= zoom_start) & (raw_time <= zoom_end)
        
        plt.figure(figsize=(14, 4))
        plt.plot(raw_time[zoom_mask], filtered[zoom_mask], color='#aaaaaa', lw=0.6, label='sEMG filtered')
        plt.plot(raw_time[:len(smooth)][zoom_mask[:len(smooth)]], 
                 env_scaled[zoom_mask[:len(smooth)]], color='orange', lw=1.5, label='RMS envelope')
        
        # Mark peaks trong zoom window
        peak_times = [raw_time[p] for p in peaks if p < len(raw_time) and zoom_start <= raw_time[p] <= zoom_end]
        peak_vals = [env_scaled[p] if p < len(env_scaled) else 0 for p in peaks if p < len(raw_time) and zoom_start <= raw_time[p] <= zoom_end]
        if peak_times:
            plt.scatter(peak_times, peak_vals, color='red', s=50, zorder=5, label='Peaks')
        
        plt.title(f'Zoom {zoom_start}-{zoom_end}s — S{subject_demo} T{trial_demo} — verify 1 peak = 1 rep')
        plt.xlabel('Time (s)'); plt.ylabel('sEMG (a.u.)')
        plt.legend(); plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # 2. Check thêm 1-2 trial "còn sống sót" khác có N cao nhưng dưới fence
        #    (các trial thực sự được dùng để train, không bị loại outlier)
        print("\n--- Top survivors (high N_total, còn trong dataset sau IQR filter) ---")
        if len(df) > 0:
            survivors = df.groupby(['subject', 'trial', 'muscle']).size().reset_index(name='n_reps')
            survivors_sorted = survivors.sort_values('n_reps', ascending=False).head(5)
            print(survivors_sorted.to_string(index=False))
        else:
            print("df trống, không có dữ liệu để kiểm tra survivors")
    else:
        print(f"Segmentation failed cho S{subject_demo} T{trial_demo}")
else:
    print("Không thể tải file CSV")

In [ ]:
# ================================================================
# Rep Interval Analysis — Chi tiết hơn cho Complex trials
# ================================================================

def rep_interval_stats_v2(segments, raw_time, n_warmup_reps=5):
    """So sánh N rep đầu tiên với phần còn lại — đảm bảo đủ mẫu ở cả 2 nhóm."""
    peak_times = [raw_time[e] for s, e in segments if e < len(raw_time)]
    if len(peak_times) < n_warmup_reps + 5:  # cần đủ dữ liệu ở "rest" để so sánh có ý nghĩa
        return None
    
    intervals = np.diff(peak_times)
    early = intervals[:n_warmup_reps - 1]   # n_warmup_reps peak đầu -> n_warmup_reps-1 khoảng
    rest  = intervals[n_warmup_reps - 1:]
    
    return {
        'n_early': len(early), 'n_rest': len(rest),
        'mean_early': early.mean(), 'std_early': early.std(),
        'mean_rest': rest.mean(), 'std_rest': rest.std(),
        'ratio': rest.mean() / early.mean() if early.mean() > 0 else np.nan,
    }

# ✅ BƯỚC 1: Tạo danh sách survivors (trials còn sống sót sau IQR filter)
survivors_high_N = df.groupby(['subject', 'trial', 'muscle']).size().reset_index(name='n_reps')
survivors_high_N = survivors_high_N.sort_values('n_reps', ascending=False).head(10)
print("Top 10 survivors (high N_total):")
print(survivors_high_N.to_string(index=False))

# ✅ BƯỚC 2: Chạy analysis cho từng survivor trial
print("\n--- Rep Interval Analysis (warmup 10s vs rest) ---")
for _, row in survivors_high_N.iterrows():
    subj, trial_n = int(row['subject']), int(row['trial'])
    path_dir = os.path.join(dir, f'subject_{subj}')
    f_csv = f'trial_{trial_n}.csv'
    
    if not os.path.exists(os.path.join(path_dir, f_csv)):
        print(f"S{subj} T{trial_n}: FILE NOT FOUND"); continue
    
    # Load & process signal
    idx = get_prime_mover(trial_n)
    raw_data = pd.read_csv(os.path.join(path_dir, f_csv)).values
    raw_time = raw_data[:, time_index[idx]].astype(float)
    raw_emg = raw_data[:, EMG_index[idx]].astype(float)
    
    b, a = signal.butter(4, [20, 450], btype='bandpass', fs=sample_freq)
    filtered = signal.filtfilt(b, a, np.nan_to_num(raw_emg, nan=0.0))
    
    # Segment reps
    result = segment_reps(filtered, raw_time, sample_freq)
    # Chạy lại đúng S8 T11 hai lần liên tiếp, so sánh segments có giống hệt nhau không
    result1 = segment_reps(filtered, raw_time, sample_freq)
    result2 = segment_reps(filtered, raw_time, sample_freq)  # cùng input, gọi lại
    print("Reproducible:", result1[0] == result2[0])  # so sánh segments list
    if not result or len(result) < 4:
        print(f"S{subj} T{trial_n}: Segmentation failed"); continue
    
    segments, _, _, _ = result
    stats = rep_interval_stats_v2(segments, raw_time, n_warmup_reps=5)
    
    # In kết quả
    if not np.isnan(stats['mean_early']) and not np.isnan(stats['mean_rest']):
        ratio = stats['mean_rest'] / stats['mean_early']
        print(f"S{subj} T{trial_n} ({row['muscle'][:20]}): "
              f"early={stats['mean_early']:.2f}s (n={stats['n_early']}) "
              f"vs rest={stats['mean_rest']:.2f}s (ratio={ratio:.1f}x)")
    else:
        print(f"S{subj} T{trial_n}: Insufficient data for interval analysis")

In [ ]:
# === BƯỚC 1: Chạy segment_reps() 2 lần, so sánh bit-exact ===
import hashlib, json

def hash_segments(segments, peaks, valleys):
    """Tạo hash từ output segment_reps() để so sánh nhanh."""
    data = json.dumps({
        'segments': [(int(s), int(e)) for s, e in segments],
        'peaks': [int(p) for p in peaks],
        'valleys': [int(v) for v in valleys],
    })
    return hashlib.md5(data.encode()).hexdigest()

# Chọn 5 trial đại diện (mix uni + complex, subject khác nhau)
TEST_CASES = [
    (1, 5),   # Subject 1, Trial 5 (R Biceps, uni)
    (8, 11),  # Subject 8, Trial 11 (R Deltoid Post C, complex - trial nghi ngờ)
    (5, 5),   # Subject 5, Trial 5 (R Biceps, uni)
    (13, 3),  # Subject 13, Trial 3 (R Deltoid Post, uni)
    (3, 9),   # Subject 3, Trial 9 (R Deltoid Ant C, complex)
]

results_run1 = {}
results_run2 = {}

for subj, trial_n in TEST_CASES:
    # Load raw data
    path_dir = os.path.join(dir, f'subject_{subj}')
    raw = pd.read_csv(os.path.join(path_dir, f'trial_{trial_n}.csv')).values
    idx = get_prime_mover(trial_n)
    raw_time = raw[:, time_index[idx]]
    raw_emg  = raw[:, EMG_index[idx]]
    b, a = signal.butter(4, [20, 450], btype='bandpass', fs=sample_freq)
    filtered = signal.filtfilt(b, a, np.nan_to_num(raw_emg, nan=0.0))

    # Run 1
    r1 = segment_reps(filtered, raw_time, sample_freq)
    results_run1[(subj, trial_n)] = {
        'hash_segments': hash_segments(r1[0], r1[1], r1[2]),
        'n_reps': len(r1[0])
    }
    # Run 2 (exact same input)
    r2 = segment_reps(filtered, raw_time, sample_freq)
    results_run2[(subj, trial_n)] = {
        'hash_segments': hash_segments(r2[0], r2[1], r2[2]),
        'n_reps': len(r2[0])
    }

    match = '✅' if results_run1[(subj, trial_n)] == results_run2[(subj, trial_n)] else '❌ MISMATCH'
    n1, n2 = len(r1[0]), len(r2[0])
    print(f"  S{subj}T{trial_n}: Run1={n1} reps, Run2={n2} reps — {match}")

In [ ]:
# === BƯỚC 2: So sánh với per_cycle_features.csv đã lưu ===
# BUG FIX: len(obj) cũ luôn = 2 (dict size), vô nghĩa.
# hash_segments được tính nhưng chưa bao giờ được dùng để so sánh.
df_saved = pd.read_csv(per_cycle_path)

all_match = True
for subj, trial_n in TEST_CASES:
    saved_n  = df_saved[df_saved['subject'] == subj].groupby('trial')['N_total'].first()
    obj      = results_run1[(subj, trial_n)]
    run_hash = obj['hash_segments']   # MD5 fingerprint của (segments, peaks, valleys)
    run_n    = obj['n_reps']
    print(f'  S{subj}T{trial_n}: hash={run_hash[:16]}... n_reps={run_n}')
    if trial_n in saved_n.index:
        csv_n    = saved_n[trial_n]
        count_ok = (csv_n == run_n)
        icon     = 'V' if count_ok else 'X'
        status   = 'OK' if count_ok else f'MISMATCH csv={csv_n} run={run_n}'
        print(f'  [{icon}] S{subj}T{trial_n}: csv_N={csv_n}, run_N={run_n} -- {status}')
        if not count_ok:
            all_match = False
    else:
        print(f'  [?] S{subj}T{trial_n}: trial not found in CSV')

print()
print('PASS: CSV sync OK' if all_match else 'FAIL: CSV OUTDATED -- regenerate per_cycle_features.csv')

In [ ]:
# === BƯỚC 3: Kiểm tra get_prime_mover() có deterministic không ===
for trial_n in [3, 5, 7, 9, 11]:
    idx1 = get_prime_mover(trial_n)
    idx2 = get_prime_mover(trial_n)
    print(f"  Trial {trial_n}: idx1={idx1}, idx2={idx2} — "
          f"{'✅' if idx1 == idx2 else '❌ NON-DETERMINISTIC'}")

In [ ]:
# Chạy lại đúng pipeline đã cho 16.8%:
# 1. Load per_cycle_features.csv (đã verify ở #1)
# 2. IQR filter
# 3. Lọc 3 cơ sạch
# 4. F1 + slope5
# 5. run_loso RF → RMSE F0, RMSE F1+slope5
# 6. (F0 - F1_slope5) / F0 >= 0.15?

clean_muscles = ['L DELTOID ANTERIOR', 'R DELTOID MEDIUS', 'L BICEPS BRACHII']
df_clean = df[df['muscle'].isin(clean_muscles)]

F1_SLOPE5 = F1_COLS + ['MNF_mean_N1_slope5']

rf_f0_clean = cached_run_loso('verify_rf_f0_clean', RandomForestRegressor, F0_COLS,
    TARGET_FCF, df_clean, {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})
rf_f1s5_clean = cached_run_loso('verify_rf_f1s5_clean', RandomForestRegressor, F1_SLOPE5,
    TARGET_FCF, df_clean, {'n_estimators':200,'max_depth':10,'n_jobs':-1,'random_state':42})

improvement = (rf_f0_clean['rmse'].mean() - rf_f1s5_clean['rmse'].mean()) / rf_f0_clean['rmse'].mean()
print(f"Re-verification: {improvement*100:.1f}% — {'GO ✅' if improvement >= 0.15 else 'NO-GO ❌'}")

In [ ]:
# Kiểm tra: RMSE improvement có tập trung ở vùng rep cao (>25) hay dàn đều?
df_clean['rep_zone'] = pd.cut(df_clean['rep_idx'], bins=[0, 25, 1000], 
                                labels=['anaerobic_zone(<=25)', 'aerobic_zone(>25)'])

for zone, grp in df_clean.groupby('rep_zone'):
    print(f"{zone}: n={len(grp)}, {grp['N_total'].describe()}")

# Nếu đa số reps rơi vào aerobic_zone → cần đánh giá lại RMSE riêng cho từng zone
# So sánh RMSE F0 vs F1(N1) TÁCH RIÊNG theo 2 zone thay vì gộp toàn bộ trial

In [ ]:
# Kiểm tra nhanh: CSV hiện tại có khớp N_total với segment_reps() live không?
# Chạy lại đúng Bước 2 đã fix (không phải qua hash, đo len() thật)
CSV_PATH = os.path.join(dir, r'D:\smilecow\HUST\Nam_3\nckh\data\processed\per_cycle_features.csv')
df_saved = pd.read_csv(CSV_PATH)
print(f"Timestamp CSV: {os.path.getmtime(CSV_PATH)}")
print(f"Tổng reps 3 cơ sạch trong CSV hiện tại: "
      f"{len(df_saved[df_saved['muscle'].isin(clean_muscles)])}")
# So với 1966 (baseline cũ) — nếu khác, xác nhận CSV đã bị regenerate và từ commit/script nào